In [ ]:
import os
import itertools
import textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from collections import defaultdict
from types import SimpleNamespace
import wandb
from scipy.stats import gmean, t as student_t
from IPython.display import display

In [ ]:
# Results come from Weights & Biases. Point these at the project the
# experiments were logged to; the server address and credentials come from the
# usual wandb configuration (the WANDB_BASE_URL environment variable or
# ~/.config/wandb/settings).
WANDB_PROJECT = "frwo"
WANDB_ENTITY = None  # None uses the default entity of your wandb login

# Every run of one execution of FRWO_Experiments.py shares a
# SESSION_ID. Only one session is ever loaded, so a plot can never mix results
# from two executions of the training script. Leave this None to use the most
# recent execution, or set it to a session id (e.g. "20260826-010125-3f9ab2")
# to reproduce an earlier set of figures exactly.
WANDB_SESSION = None

# Every run records the experiment config it came from
# (configs/experiments/<name>.yaml) as CONFIG_NAME, and one session can run
# several. Only this config's runs are loaded, so a plot never mixes two
# configs, and with WANDB_SESSION None the most recent session that ran this
# config is loaded. None loads every run of the session, for sessions logged
# before experiment configs had names.
WANDB_CONFIG = "main"

# Fairness Reward Weight Optimization's training adjustments
# (SUBDOM_WEIGHT_ADJUST_LIST and MANUAL_WEIGHT_ADJUST_LIST) each get W&B runs
# of their own, which record their entry as SUBDOM_WEIGHT_ADJUST or
# MANUAL_WEIGHT_ADJUST. None loads only the runs without one. Naming one, as
# its key and entry, loads its runs in place of the runs they adjust (those of
# the same weight adjustment without one), e.g.
#   ("SUBDOM_WEIGHT_ADJUST", (("DemPar", 2.0),))
#   ("MANUAL_WEIGHT_ADJUST", (("DemPar", 0.1),))
WANDB_TRAINING_ADJUST = None
TRAINING_ADJUST_KEYS = ("SUBDOM_WEIGHT_ADJUST", "MANUAL_WEIGHT_ADJUST")
# Training adjustments the training script no longer has. Sessions logged
# before one was removed still hold its runs, which are left out entirely, so
# that they are never mistaken for the runs they adjust.
RETIRED_TRAINING_ADJUST_KEYS = ("ALPHA_ADJUST",)


def to_tuple(data):
    if isinstance(data, list):
        return tuple(to_tuple(i) for i in data)
    elif isinstance(data, dict):
        return {k: to_tuple(v) for k, v in data.items()}
    else:
        return data


def run_bias_type(config):
    """The bias type a run covers: a dataset or a demo bias type, or `()`.

    Sessions logged before demo bias types existed record only their dataset
    bias type, as DATASET_BIAS_TYPE.
    """
    return config.get("BIAS_TYPE", config["DATASET_BIAS_TYPE"])


def run_training_adjust(config):
    """The training adjustment a run covers, as `(key, entry)`, or None.

    Sessions logged before training adjustments existed have none.
    """
    for key in TRAINING_ADJUST_KEYS:
        if config.get(key):
            return (key, config[key])
    return None


def select_training_adjust(records, training_adjust):
    """The records of the runs without a training adjustment, with those of
    `training_adjust` (see WANDB_TRAINING_ADJUST) in place of the runs they
    adjust."""
    def adjusted_run_key(config):
        return (
            config["DATASET"],
            config["EXPERT_ALGO"],
            config.get("ALGORITHM", "Fairness Reward Weight Optimization"),
            run_bias_type(config),
            config["WEIGHT_ADJUST"],
            config.get("TRIAL", 0),
        )

    selected = [
        record
        for record in records
        if run_training_adjust(record["config"]) in (None, training_adjust)
    ]
    adjusted = {
        adjusted_run_key(record["config"])
        for record in selected
        if run_training_adjust(record["config"]) is not None
    }
    return [
        record
        for record in selected
        if run_training_adjust(record["config"]) is not None
        or adjusted_run_key(record["config"]) not in adjusted
    ]


class ResultsLookup(dict):
    """Nested results dict that names the missing configuration on a miss.

    The plotting cells below index five levels deep, e.g.
    `averaged_data[dataset][expert][algorithm][dataset_bias_type][weight_adjust]`.
    With a plain defaultdict a configuration the session never ran would
    silently produce an empty frame, and the failure only surfaced later as a
    confusing
    `KeyError: 'sum_abs_subdominance_val'` from `.loc`. This reports which
    configuration is actually absent, and what the session does contain.
    """

    LEVELS = ("dataset", "expert", "algorithm", "dataset_bias_type", "weight_adjust")

    def __init__(self, depth=0, path=(), session_id=""):
        super().__init__()
        self._depth = depth
        self._path = path
        self._session_id = session_id

    def child(self, key):
        """Get or create the sub-lookup for `key` (never triggers __missing__)."""
        return self.setdefault(
            key,
            ResultsLookup(self._depth + 1, self._path + (key,), self._session_id),
        )

    def __missing__(self, key):
        level = self.LEVELS[self._depth]
        context = ", ".join(f"{n}={v!r}" for n, v in zip(self.LEVELS, self._path))
        raise KeyError(
            f"No results for {level}={key!r}"
            + (f" under {context}" if context else "")
            + f" in W&B session {self._session_id!r}."
            f" Available {level} values there: {sorted(self.keys(), key=repr)}."
            " Either re-run the training script with a config that produces it,"
            " or set WANDB_SESSION (or WANDB_CONFIG) to a session that has it."
        )


def config_filter(config_name):
    """The W&B run filter that selects the runs of experiment config `config_name`."""
    return {} if config_name is None else {"config.CONFIG_NAME": config_name}


def resolve_session(api, path, session=None, config_name=None):
    """Return the session id to load: `session`, or the most recent one that
    ran experiment config `config_name`."""
    if session is not None:
        return session

    newest = next(
        iter(
            api.runs(
                path,
                filters=config_filter(config_name) or None,
                order="-created_at",
                per_page=1,
            )
        ),
        None,
    )
    if newest is None:
        raise RuntimeError(
            f"No runs found in W&B project {path!r}"
            + ("" if config_name is None else f" for experiment config {config_name!r}")
            + ". Run FRWO_Experiments.py first."
        )
    return newest.config["SESSION_ID"]


def read_session_runs(project, entity=None, session=None, config_name=None):
    """Read the runs of a single execution of the training script, of
    experiment config `config_name` (or of every config, if None).

    Each run is one trial of one algorithm, one dataset bias type and one
    weight adjustment. Its config holds the experiment parameters and its
    summary holds that trial's results.

    Returns
    -------
    session_id : str
        The session that was loaded.
    records : list<dict>
        One record per run, with its `created_at`, `config`, `metrics` and
        `demo_losses`.
    """
    api = wandb.Api()
    path = f"{entity}/{project}" if entity else project
    session_id = resolve_session(api, path, session, config_name)

    records = []
    for run in api.runs(
        path,
        filters={
            "state": "finished",
            "config.SESSION_ID": session_id,
            **config_filter(config_name),
        },
    ):
        summary = dict(run.summary)
        # Trials that did not converge have no results to plot.
        if not summary.get("converged", False):
            continue

        # Config values come back from W&B as lists; the plotting cells index
        # with tuple literals, so they have to be tuples to be usable as keys.
        config = to_tuple(
            {k: v for k, v in run.config.items() if not k.startswith("_")}
        )
        if any(config.get(key) for key in RETIRED_TRAINING_ADJUST_KEYS):
            continue
        metrics = {
            k: v
            for k, v in summary.items()
            if not k.startswith("_")
            and isinstance(v, (int, float))
            and not isinstance(v, bool)
        }

        # The expert's and the model's per-demonstration losses are lists (one
        # row per subdominance group), not scalars, so the filter above drops
        # them. They are kept apart: they have no meaningful average across
        # trials. Sessions logged before `model_demo_feat_loss_*` was recorded
        # only have the expert's.
        demo_losses = {
            k: v
            for k, v in summary.items()
            if k.startswith(("expert_demo_feat_loss_", "model_demo_feat_loss_"))
        }

        records.append(
            {
                "created_at": run.created_at,
                "config": config,
                "metrics": metrics,
                "demo_losses": demo_losses,
            }
        )

    return session_id, records


def aggregate_records(session_id, records):
    """Collect the trials of each (dataset, expert, algorithm, dataset bias
    type, weight adjustment) configuration of `records`, all from the one
    session `session_id`.

    Each run covers exactly one algorithm, one bias type and one weight
    adjustment, so the latter two are single tuples -- `()` for the unbiased
    dataset and for the unadjusted weights. Sessions logged before the
    Superhuman Fairness baseline existed have no ALGORITHM in their config;
    those runs are all Fairness Reward Weight Optimization ones.

    Returns
    -------
    results : SimpleNamespace
        `session_id`; `averaged_data`, every configuration's results averaged
        over its trials; `per_trial_data`, the same results un-averaged -- one
        row per trial, indexed by TRIAL -- for plots that need the spread
        across trials (e.g. error bars); `averaged_info`, every
        configuration's run config; and `expert_demo_losses` and
        `model_demo_losses`, read through `get_expert_demo_losses()` and
        `get_model_demo_losses()` below.
    """
    trials_by_config = defaultdict(list)
    trial_ids_by_config = defaultdict(list)
    config_by_key = {}
    first_seen = {}
    # The expert's demonstrations, as losses per subdominance group, keyed by
    # (dataset, expert, dataset bias type, trial). Every algorithm of one bias
    # type and trial is scored against the same groups, so any one of their
    # runs serves.
    expert_demo_losses = {}
    # The model's losses on those same groups, keyed by its configuration and
    # trial.
    model_demo_losses = {}
    for record in records:
        config = record["config"]
        config_key = (
            config["DATASET"],
            config["EXPERT_ALGO"],
            config.get("ALGORITHM", "Fairness Reward Weight Optimization"),
            run_bias_type(config),
            config["WEIGHT_ADJUST"],
        )
        trial = config.get("TRIAL", 0)
        trials_by_config[config_key].append(record["metrics"])
        trial_ids_by_config[config_key].append(trial)
        config_by_key[config_key] = config
        demo_losses = record["demo_losses"]
        if "expert_demo_feat_loss_metrics" in demo_losses:
            demo_key = (config["DATASET"], config["EXPERT_ALGO"], run_bias_type(config), trial)
            expert_demo_losses.setdefault(demo_key, demo_losses)
            if "model_demo_feat_loss_test" in demo_losses:
                model_demo_losses[config_key + (trial,)] = demo_losses
        first_seen[config_key] = min(
            first_seen.get(config_key, record["created_at"]), record["created_at"]
        )

    averaged_data = ResultsLookup(session_id=session_id)
    per_trial_data = ResultsLookup(session_id=session_id)
    averaged_info = {}
    # Oldest configuration first, so that datasets appear in the order they were
    # run, which is the order the plots lay them out along the x-axis.
    for config_key in sorted(trials_by_config, key=lambda k: first_seen[k]):
        dataset, expert, algorithm, dataset_bias_type, weight_adjust = config_key
        trial_data = pd.DataFrame(
            trials_by_config[config_key],
            index=pd.Index(trial_ids_by_config[config_key], name="TRIAL"),
        )
        averaged_data.child(dataset).child(expert).child(algorithm).child(
            dataset_bias_type
        )[weight_adjust] = trial_data.mean()
        per_trial_data.child(dataset).child(expert).child(algorithm).child(
            dataset_bias_type
        )[weight_adjust] = trial_data
        averaged_info.setdefault(dataset, {}).setdefault(expert, {}).setdefault(
            algorithm, {}
        ).setdefault(dataset_bias_type, {})[weight_adjust] = config_by_key[config_key]

    return SimpleNamespace(
        session_id=session_id,
        averaged_data=averaged_data,
        per_trial_data=per_trial_data,
        averaged_info=averaged_info,
        expert_demo_losses=expert_demo_losses,
        model_demo_losses=model_demo_losses,
    )


def describe_results(results):
    """Print the session `results` came from and every configuration in it."""
    configurations = [
        (dataset, expert, algorithm, dataset_bias_type, weight_adjust, trials)
        for dataset, by_expert in results.per_trial_data.items()
        for expert, by_algorithm in by_expert.items()
        for algorithm, by_bias_type in by_algorithm.items()
        for dataset_bias_type, by_weight_adjust in by_bias_type.items()
        for weight_adjust, trials in by_weight_adjust.items()
    ]
    print(f"W&B session:  {results.session_id}")
    print(
        f"runs loaded:  {sum(len(trials) for *_, trials in configurations)}"
        f" across {len(configurations)} configurations"
    )
    for dataset, expert, algorithm, dataset_bias_type, weight_adjust, trials in configurations:
        print(
            f"  {dataset} / {expert} / {algorithm} / bias={dataset_bias_type}"
            f" / weights={weight_adjust}: {len(trials)} trial(s)"
        )


def load_results(config_name, session=None, training_adjust=None):
    """The results of experiment config `config_name` in W&B session
    `session` (or its most recent one), with the runs of `training_adjust` (see
    WANDB_TRAINING_ADJUST) in place of the runs they adjust."""
    session_id, records = read_session_runs(WANDB_PROJECT, WANDB_ENTITY, session, config_name)
    records = select_training_adjust(records, training_adjust)
    if not records:
        raise RuntimeError(
            f"W&B session {session_id!r} has no finished, converged runs"
            + ("" if config_name is None else f" of experiment config {config_name!r}")
            + ". Pick another session or config, or re-run the training script."
        )
    results = aggregate_records(session_id, records)
    describe_results(results)
    return results


main_results = load_results(WANDB_CONFIG, WANDB_SESSION, WANDB_TRAINING_ADJUST)
session_id = main_results.session_id
averaged_data = main_results.averaged_data
per_trial_data = main_results.per_trial_data
averaged_info = main_results.averaged_info
expert_demo_losses = main_results.expert_demo_losses
model_demo_losses = main_results.model_demo_losses


def get_expert_demo_losses(
    dataset, expert, dataset_bias_type, trial=0, split="train", results=None
):
    """The expert's demonstrations of one trial, as losses per subdominance group.

    `results` is a `load_results()` result; None reads the main config's.

    Returns
    -------
    losses : pandas.DataFrame
        One row per subdominance group (i.e. per demonstration), one column per
        subdominance metric, holding that metric as a loss (lower is better).
    """
    results = main_results if results is None else results
    key = (dataset, expert, dataset_bias_type, trial)
    if key not in results.expert_demo_losses:
        raise KeyError(
            f"No expert demonstration losses for dataset={dataset!r},"
            f" expert={expert!r}, dataset_bias_type={dataset_bias_type!r},"
            f" trial={trial!r} in W&B session {results.session_id!r}. Sessions"
            " logged before these were recorded do not have them; re-run the"
            " training script. Available: "
            f"{sorted(results.expert_demo_losses, key=repr)}."
        )
    stored = results.expert_demo_losses[key]
    return pd.DataFrame(
        np.asarray(stored[f"expert_demo_feat_loss_{split}"], dtype=float),
        columns=list(stored["expert_demo_feat_loss_metrics"]),
    )


def get_model_demo_losses(
    dataset,
    expert,
    algorithm,
    dataset_bias_type,
    weight_adjust,
    trial=0,
    split="test",
    results=None,
):
    """A model's losses of one trial on the expert's subdominance groups.

    Row `i` holds the model's decisions on exactly the rows of the expert's
    group `i` (see `get_expert_demo_losses()`), so that the two can be compared
    group by group, as subdominance compares them. A demo bias type reuses the
    unbiased run's groups, so its rows also line up with the unbiased run's
    expert demonstrations of the same trial.

    Returns
    -------
    losses : pandas.DataFrame or None
        One row per subdominance group, one column per subdominance metric.
        None for a session logged before these were recorded.
    """
    results = main_results if results is None else results
    key = (dataset, expert, algorithm, dataset_bias_type, weight_adjust, trial)
    stored = results.model_demo_losses.get(key)
    if stored is None:
        return None
    return pd.DataFrame(
        np.asarray(stored[f"model_demo_feat_loss_{split}"], dtype=float),
        columns=list(stored["expert_demo_feat_loss_metrics"]),
    )

In [ ]:
selected_expert = "OptClfMDPPol"
# selected_expert = "PostProcDemo"

# Which technique the plots below are drawn for. Every run carries its
# technique as `ALGORITHM`, so switching this re-draws the same figures for
# another one; a comparison plot can index several, e.g.
# `averaged_data[dataset][selected_expert]["Superhuman Fairness"][bias][()]`.
# Valid values are the entries of the training script's ALGORITHMS list:
# "Fairness Reward Weight Optimization", "Superhuman Fairness",
# "Superhuman Fairness Fixed", "Superhuman Fairness Neural Network",
# "Superhuman Fairness Neural Network Fixed",
# "Post Proc DP", "Post Proc EqOdds",
# "Fair LogLoss DP" and "Fair LogLoss EqOdds".
selected_algorithm = "Fairness Reward Weight Optimization"

# Each W&B run covers exactly one bias type and one weight adjustment, so every
# key below is a single tuple, matching one entry of the training script's
# DATASET_BIAS_TYPE_LIST or DEMO_BIAS_TYPE_LIST / WEIGHT_ADJUST_LIST -- plus the
# `()` that is always run for the unbiased dataset and for the unadjusted
# weights. (The `dataset_bias_type` level of the lookups below holds either
# variety of bias type.) Only
# configurations the loaded session actually ran can be selected here; every
# technique other than Fairness Reward Weight Optimization has no reward weights to adjust,
# so those only ever report the unadjusted `()` one.
unbiased_types = [
    (),
]
biased_types = [
    # ("dataset_unbalanced_redlining", 0.2),
    # ("dataset_balanced_redlining", 0.2),
    # ("dataset_perfectly_balanced_redlining", 0.2),
    # ("dataset_noise_bias", 0.2, True, True),
    # ("dataset_corruption_bias", "CatBoost", 0.01, "gaussian", 1.0),
    # ("demo_unbalanced_redlining", 0.2),
    # ("demo_balanced_redlining", 0.2),
    # ("demo_perfectly_balanced_redlining", 0.2),
    # ("demo_noise_bias", 0.2),
    # ("demo_corruption_bias", "CatBoost", 0.01, "gaussian", 1.0),
]

# Fairness Reward Weight Optimization is the only technique with reward weights to adjust, so
# it is the only one with runs for any weight adjustment other than `()`.
weight_adjust_algorithm = "Fairness Reward Weight Optimization"

unadjusted_weights = ()
# Each dataset's `opt_debias` optimizer is tuned separately (its
# WEIGHT_ADJUST_LIST in configs/tuned_hyperparameters.yaml), so the plots
# below are not given one weight adjustment for every dataset: each dataset is
# plotted with the one its own runs used, read from the loaded session by
# `tuned_weight_adjust()`. A session whose runs of one dataset cover several
# weight adjustments, or none (an experiment config with RUN_OPT_DEBIAS off),
# has no single one to use; name the one to plot for each such dataset here,
# as one entry of its WEIGHT_ADJUST_LIST or `()` for the unadjusted weights,
# e.g.
#   ("opt_debias", "optuna", "GP", 200)
#   ("opt_debias", "pybobyqa", "Multi-Start BOBYQA", 200)
weight_adjust_override = {
    # "COMPAS": ("opt_debias", "optuna", "GP", 200),
}


def tuned_weight_adjust(dataset, expert, bias_type):
    """The (non-`()`) weight adjustment the session's Fairness Reward Weight Optimization
    runs of `dataset` used, or its entry in `weight_adjust_override`."""
    if dataset in weight_adjust_override:
        return weight_adjust_override[dataset]
    ran = [
        weight_adjust
        for weight_adjust in averaged_data[dataset][expert][weight_adjust_algorithm][bias_type]
        if weight_adjust != ()
    ]
    if len(ran) != 1:
        raise ValueError(
            f"Expected exactly one adjusted weight configuration for dataset={dataset!r},"
            f" expert={expert!r}, bias_type={bias_type!r} in W&B session {session_id!r},"
            f" found {ran}. Name the one to plot in `weight_adjust_override`."
        )
    return ran[0]


def weight_adjust_label(weight_adjust):
    """E.g. `optuna GP (200)` for `("opt_debias", "optuna", "GP", 200)`."""
    if not weight_adjust:
        return "unadjusted"
    _, library, optimizer, budget = weight_adjust
    return f"{library} {optimizer} ({budget})"


def session_algorithms(expert):
    """Every technique the session ran for `expert`, Fairness Reward Weight Optimization first."""
    algorithms = []
    for dataset in averaged_data:
        for algorithm in averaged_data[dataset].get(expert, {}):
            if algorithm not in algorithms:
                algorithms.append(algorithm)
    return sorted(algorithms, key=lambda algorithm: algorithm != weight_adjust_algorithm)


def model_weight_adjust(algorithm, dataset, expert, bias_type):
    """The weight adjustment whose results stand for `algorithm` on `dataset`: the tuned one
    for Fairness Reward Weight Optimization, the unadjusted `()` for every other technique.
    None when the session did not run `algorithm` on `dataset` under `bias_type`."""
    if bias_type not in averaged_data[dataset].get(expert, {}).get(algorithm, {}):
        return None
    if algorithm == weight_adjust_algorithm:
        return tuned_weight_adjust(dataset, expert, bias_type)
    return unadjusted_weights

In [ ]:
# Shared Plotting Setup:

# Where every table and figure below is saved, the look they share, and the helpers they are drawn
# with. Run this cell after the selection cell above, and before any cell below.
#
# Every output goes under OUTPUT_DIR, in `exp_tables/<config>/` or `exp_plots/<config>/`, one
# folder per experiment config, so that each config's outputs are kept.
#
# The colors are those of a validated palette, chosen so that every pair stays distinguishable
# under the common color vision deficiencies:
# * Fairness Reward Weight Optimization is always blue (with its equal, unadjusted weights a lighter blue), the
#   Superhuman Fairness variants orange and the other fair-classification baselines aqua; within a
#   family, a model is told apart by its marker. A scatter plot can only keep three hues apart,
#   which is why the baselines share them by family.
# * The expert is always neutral gray: it is the reference every model is compared against.
# * A series of experts or datasets takes the categorical colors in their fixed order.
# * Differences are drawn on a diverging red-blue scale with a neutral midpoint: blue where the
#   model is better than its reference, red where it is worse.
#
# Figures embed their fonts as TrueType (`pdf.fonttype` 42): the ACM camera-ready checks reject the
# Type 3 fonts matplotlib embeds by default.

OUTPUT_DIR = "./../../experiment_output/frwo"
# False only saves the figures, without also drawing them in the notebook.
SHOW_FIGURES = True

INK = "#0b0b0b"
INK_SECONDARY = "#52514e"
MUTED = "#898781"
GRIDLINE = "#e1e0d9"
BASELINE = "#c3c2b7"
SURFACE = "#ffffff"
SERIES_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
EXPERT_COLOR = MUTED
FRWO_COLOR = SERIES_COLORS[0]
FRWO_UNADJUSTED_COLOR = "#86b6ef"  # a lighter step of the same blue
DIVERGING_CMAP = plt.get_cmap("RdBu").copy()
DIVERGING_CMAP.set_bad(SURFACE)

plt.rcParams.update(
    {
        "figure.dpi": 110,
        "savefig.dpi": 300,
        "savefig.bbox": "tight",
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "font.size": 8,
        "axes.titlesize": 9,
        "axes.labelsize": 8,
        "xtick.labelsize": 7,
        "ytick.labelsize": 7,
        "legend.fontsize": 7,
        "text.color": INK,
        "axes.labelcolor": INK_SECONDARY,
        "xtick.color": INK_SECONDARY,
        "ytick.color": INK_SECONDARY,
        "axes.edgecolor": BASELINE,
        "axes.linewidth": 0.8,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "axes.axisbelow": True,
        "grid.color": GRIDLINE,
        "grid.linewidth": 0.6,
        "grid.linestyle": "-",
        "legend.frameon": False,
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "savefig.facecolor": SURFACE,
    }
)

# Each technique's short name, color and marker. A technique without an entry is drawn in gray.
ALGORITHM_STYLE = {
    "Fairness Reward Weight Optimization": ("FRWO", SERIES_COLORS[0], "o"),
    "Superhuman Fairness": ("SH", SERIES_COLORS[1], "s"),
    "Superhuman Fairness Fixed": ("SH-Fixed", SERIES_COLORS[1], "D"),
    "Superhuman Fairness Neural Network": ("SH-NN", SERIES_COLORS[1], "^"),
    "Superhuman Fairness Neural Network Fixed": ("SH-NN-Fixed", SERIES_COLORS[1], "v"),
    "Post Proc DP": ("PP-DP", SERIES_COLORS[2], "P"),
    "Post Proc EqOdds": ("PP-EqOdds", SERIES_COLORS[2], "X"),
    "Fair LogLoss DP": ("FLL-DP", SERIES_COLORS[2], "<"),
    "Fair LogLoss EqOdds": ("FLL-EqOdds", SERIES_COLORS[2], ">"),
}
# The objectives every feature expectation figure reports, in order. FPRPar and FNRPar are left
# out because they are effectively the same as EqOpp and TNRPar.
measurable_metrics = ["Acc", "AccPar", "DemPar", "EqOpp", "TNRPar"]

# The expert a model is compared against, under a biased run: the biased demonstrations it learned
# from ("demos"), or the expert's unbiased demonstrations ("unbiased"). The first shows whether the
# model dominates the low-quality expert it imitated, the second whether it recovers the true one.
# Under the unbiased dataset both are the same expert.
REFERENCE_TITLE = {"demos": "the demonstrations learned from", "unbiased": "the unbiased expert"}


def algorithm_label(algorithm):
    return ALGORITHM_STYLE.get(algorithm, (algorithm,))[0]


def algorithm_color(algorithm):
    return ALGORITHM_STYLE.get(algorithm, (None, MUTED))[1]


def algorithm_marker(algorithm):
    return ALGORITHM_STYLE.get(algorithm, (None, None, "o"))[2]


def dataset_label(dataset):
    """E.g. `ACSIncome (MA)` for `ACSIncome__MA`."""
    name, _, region = dataset.partition("__")
    return f"{name} ({region})" if region else name


def slug(text):
    """A filename-friendly version of `text`."""
    return str(text).replace(" ", "_").replace(".", "-")


def bias_type_label(dataset_bias_type):
    """A filename-friendly name for a dataset bias type."""
    if not dataset_bias_type:
        return "unbiased"
    return "_".join(str(component) for component in dataset_bias_type)


def bias_type_short(dataset_bias_type):
    """E.g. `Balanced redlining` for `("demo_balanced_redlining", 0.2)`."""
    if not dataset_bias_type:
        return "No bias"
    name = str(dataset_bias_type[0])
    for prefix in ("demo_", "dataset_"):
        name = name.removeprefix(prefix)
    return name.removesuffix("_bias").replace("_", " ").capitalize()


def bias_type_title(dataset_bias_type):
    """E.g. `Demo balanced redlining (0.2)` for `("demo_balanced_redlining", 0.2)`."""
    if not dataset_bias_type:
        return "No bias"
    name = str(dataset_bias_type[0]).removesuffix("_bias").replace("_", " ").capitalize()
    parameters = ", ".join(str(component) for component in dataset_bias_type[1:])
    return f"{name} ({parameters})" if parameters else name


def bias_type_columns(bias_types):
    """A table column name for each of `bias_types`: its `bias_type_short()` name, or its
    `bias_type_title()` where two of them share a short name."""
    shorts = [bias_type_short(bias_type) for bias_type in bias_types]
    return {
        bias_type: short if shorts.count(short) == 1 else bias_type_title(bias_type)
        for bias_type, short in zip(bias_types, shorts)
    }


def is_demo_bias(dataset_bias_type):
    return bool(dataset_bias_type) and str(dataset_bias_type[0]).startswith("demo_")


def references_for(dataset_bias_type):
    """The experts a model trained under `dataset_bias_type` is compared against."""
    return ("demos",) if dataset_bias_type == () else ("demos", "unbiased")


def reference_suffix(dataset_bias_type, reference):
    """A filename suffix naming the reference, empty when there is only one."""
    return "" if dataset_bias_type == () else f"_vs_{reference}"


def expert_label(dataset_bias_type, reference):
    if dataset_bias_type == ():
        return "Expert"
    return "Expert (biased demos)" if reference == "demos" else "Expert (unbiased)"


def expert_mu_key(split, metric, reference):
    """The run summary key of the reference expert's feature expectation of `metric`."""
    if reference == "unbiased":
        return f"muE_{split}_unbiased_{metric}_mean"
    return f"muE_{split}_{metric}_mean"


def ci95(values):
    """The half-width of the 95% confidence interval of the mean of `values` (Student's t). 0 for
    fewer than two values, where it is undefined."""
    values = np.asarray(values, dtype=float)
    values = values[~np.isnan(values)]
    if len(values) < 2:
        return 0.0
    return student_t.ppf(0.975, len(values) - 1) * values.std(ddof=1) / np.sqrt(len(values))


def output_dir(kind, config_name, *subdirs):
    """`OUTPUT_DIR/<kind>/<config_name>/<subdirs>`, created if it does not exist. `kind` is
    `exp_tables` or `exp_plots`."""
    path = os.path.join(OUTPUT_DIR, kind, config_name or "all_configs", *subdirs)
    os.makedirs(path, exist_ok=True)
    return path


def save_figure(fig, path, show=None):
    """Save `fig` to `path`, then draw it in the notebook (if `show`, by default SHOW_FIGURES)."""
    fig.savefig(path)
    print(f"Saved {path}")
    if SHOW_FIGURES if show is None else show:
        plt.show()
    plt.close(fig)


def session_experts(data=None):
    """Every expert of `data` (by default the main config's `averaged_data`), in the order it
    first ran them."""
    data = averaged_data if data is None else data
    experts = []
    for dataset in data:
        for expert in data[dataset]:
            if expert not in experts:
                experts.append(expert)
    return experts


def session_bias_types(expert, algorithm=None, data=None):
    """Every bias type `algorithm` (by default Fairness Reward Weight Optimization) ran under
    for `expert`, the unbiased `()` first, in the order they were run."""
    data = averaged_data if data is None else data
    algorithm = weight_adjust_algorithm if algorithm is None else algorithm
    bias_types = []
    for dataset in data:
        for dataset_bias_type in data[dataset].get(expert, {}).get(algorithm, {}):
            if dataset_bias_type not in bias_types:
                bias_types.append(dataset_bias_type)
    return sorted(bias_types, key=lambda bias_type: bias_type != ())


def single_weight_adjust(by_weight_adjust, context, override=None):
    """The one weight adjustment other than `()` in `by_weight_adjust` (a configuration's
    results, keyed by weight adjustment), or `override` if one is named."""
    if override is not None:
        return override
    ran = [weight_adjust for weight_adjust in by_weight_adjust if weight_adjust != ()]
    if len(ran) != 1:
        raise ValueError(
            f"Expected exactly one adjusted weight configuration for {context}, found {ran}."
            " Name the one to use in this section's weight adjustment override."
        )
    return ran[0]


def model_rows(expert, include_unadjusted):
    """The models a table or heatmap has a row for, as `(label, algorithm, unadjusted)`: every
    technique the session ran for `expert`, and, if `include_unadjusted`, Fairness Reward Weight
    Optimization's unadjusted run (its equal reward weights, before the weight adjustment learns
    them from the demonstrations) right after its adjusted one -- the ablation that shows what the
    weight adjustment adds."""
    rows = []
    for algorithm in session_algorithms(expert):
        rows.append((algorithm_label(algorithm), algorithm, False))
        if algorithm == weight_adjust_algorithm and include_unadjusted:
            rows.append((f"{algorithm_label(algorithm)} (equal weights)", algorithm, True))
    return rows


def row_weight_adjust(algorithm, unadjusted, dataset, expert, dataset_bias_type):
    """The weight adjustment a row of `model_rows()` stands for on `dataset`, or None when the
    session has no such run. The unadjusted row is None when the tuned weight adjustment is
    itself `()`, so that it never repeats the row above it."""
    by_weight_adjust = (
        averaged_data[dataset].get(expert, {}).get(algorithm, {}).get(dataset_bias_type)
    )
    if by_weight_adjust is None:
        return None
    if algorithm == weight_adjust_algorithm and all(wa == () for wa in by_weight_adjust):
        return None if unadjusted else ()
    if unadjusted:
        return () if () in by_weight_adjust else None
    return model_weight_adjust(algorithm, dataset, expert, dataset_bias_type)


def columns_of(trials, prefix, suffix=""):
    """`trials`' columns `<prefix><metric><suffix>` of every metric of `measurable_metrics`, named
    by the metric."""
    values = trials[[f"{prefix}{metric}{suffix}" for metric in measurable_metrics]]
    values.columns = measurable_metrics
    return values


def grouped_bars(ax, categories, series, group_width=0.8):
    """Side-by-side bars on `ax`: one group per category, one bar per series in each group.

    `series` is a list of `(label, values, errors, color)`, with one value (and one error, or None
    for no error bar) per category. Returns each series' bar positions.
    """
    x = np.arange(len(categories))
    width = group_width / len(series)
    positions = []
    for i, (label, values, errors, color) in enumerate(series):
        position = x - group_width / 2 + width * (i + 0.5)
        ax.bar(position, values, width, color=color, label=label, edgecolor=SURFACE,
               linewidth=1.0, zorder=2)
        if errors is not None and np.nanmax(np.asarray(errors, dtype=float), initial=0) > 0:
            ax.errorbar(position, values, yerr=errors, fmt="none", ecolor=INK_SECONDARY,
                        elinewidth=0.8, capsize=1.5, zorder=3)
        positions.append(position)
    ax.set_xticks(x)
    ax.set_xticklabels(categories)
    ax.grid(axis="x", visible=False)
    ax.axhline(0, color=BASELINE, linewidth=0.8, zorder=2)
    return positions


def annotate_differences(ax, positions, tops, differences, scale=100, fmt="{:+.1f}"):
    """Write each difference (times `scale`) above its group of bars, whose tallest bar reaches
    `tops`."""
    for position, top, difference in zip(positions, tops, differences):
        if np.isnan(difference):
            continue
        ax.annotate(fmt.format(difference * scale), (position, top), xytext=(0, 2),
                    textcoords="offset points", ha="center", va="bottom", fontsize=6,
                    color=INK_SECONDARY)


def diverging_heatmap(ax, values, row_labels, column_labels, vmax=None, annotate=False,
                      fmt="{:+.1f}"):
    """Draw `values` (rows x columns) on the diverging scale, centered on 0. `vmax` bounds the
    scale (by default the largest absolute value). Returns the image, for a colorbar."""
    values = np.asarray(values, dtype=float)
    if vmax is None:
        vmax = np.nanmax(np.abs(values), initial=0)
    vmax = max(vmax, 1e-9)
    image = ax.imshow(np.ma.masked_invalid(values), cmap=DIVERGING_CMAP, aspect="auto",
                      norm=TwoSlopeNorm(vcenter=0, vmin=-vmax, vmax=vmax))
    ax.set_xticks(np.arange(len(column_labels)))
    ax.set_xticklabels(column_labels)
    ax.set_yticks(np.arange(len(row_labels)))
    ax.set_yticklabels(row_labels)
    ax.grid(False)
    ax.tick_params(length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)
    if annotate:
        for (row, column), value in np.ndenumerate(values):
            if np.isnan(value):
                continue
            # White text on the darkest cells, ink on the rest
            dark = abs(value) > 0.6 * vmax
            text = fmt.format(value)
            if float(text) == 0:
                text = fmt.format(0.0)  # never "-0.00"
            ax.text(column, row, text, ha="center", va="center", fontsize=6,
                    color=SURFACE if dark else INK)
    return image


def figure_title(fig, text):
    """Title `fig` with `text`, wrapped to the figure's width."""
    fig.suptitle(textwrap.fill(text, width=max(40, int(fig.get_figwidth() * 13))))


def colorbar_beside(fig, ax, image, label, extend="neither"):
    """A colorbar the full height of `ax`, just to its right."""
    colorbar = fig.colorbar(image, cax=ax.inset_axes([1.03, 0, 0.04, 1]), extend=extend)
    colorbar.set_label(label)
    colorbar.outline.set_visible(False)
    return colorbar


def shared_legend(fig, axes, ncol=None, order=None, **kwargs):
    """One legend below `fig` with every distinct label of `axes`, in the order first drawn, or
    with the labels of `order` first."""
    handles = {}
    for ax in np.ravel(axes):
        for handle, label in zip(*ax.get_legend_handles_labels()):
            handles.setdefault(label, handle)
    if order:
        handles = {
            **{label: handles[label] for label in order if label in handles},
            **{label: handle for label, handle in handles.items() if label not in order},
        }
    if handles:
        fig.legend(handles.values(), handles.keys(), loc="upper center",
                   ncol=ncol or min(len(handles), 6), bbox_to_anchor=(0.5, 0.0), **kwargs)

In [ ]:
# Overall Subdominance Results Tables:

# One table per expert the session learned from (each EXPERT_ALGO of the experiment config that it
# ran) and bias type, since every model's subdominance is measured against that expert's
# demonstrations under that bias. Each shows the sum of absolute subdominance of Fairness Reward
# Weight Optimization -- with each dataset's tuned weight adjustment -- and of every baseline
# technique the session ran, on each dataset's test split (`sum_abs_subdominance_test`), as its mean
# over the session's trials, +- its standard deviation when there are several. Lower is better.
# One row per technique, one column per dataset, and a last column with the geometric mean of each
# technique's mean results over the datasets -- left empty for a technique without a result on
# every dataset, whose mean over fewer datasets would not be comparable. With
# `table_include_unadjusted`, a row for Fairness Reward Weight Optimization's unadjusted run, with
# its equal reward weights, follows its own, to show what the weight adjustment adds. The best result in each column
# is in bold, and every result below one -- a model that, on average, dominates the expert -- is
# underlined. Both are judged on the means as printed, so that two results that print alike are
# never told apart (both are then bold), and a result that prints as 1.000 is never underlined.
#
# A summary table per expert then gives each technique's geometric mean under every bias type, one
# column per bias type, for the main text; the per-bias-type tables are the appendix's.
#
# Each table is saved as a complete LaTeX `table` environment, needing no extra LaTeX packages, to
# `subdominance_results_<expert>_<bias type>.tex` and `subdominance_summary_<expert>.tex`, displayed
# with e.g. `\input{subdominance_summary_OptClfMDPPol.tex}`. `subdominance_results.tex` holds every
# table, one after another.

# The bias types the models were trained under. None makes tables for every bias type the session
# ran, `()` (the unbiased dataset) first; or list some, e.g.
#   [(), ("demo_noise_bias", 0.2)]
table_bias_types = None

# The experts to make tables for. None makes them for every expert the session ran.
table_experts = None
# table_experts = ["OptClfMDPPol", "PostProcDemo"]

table_metric = "sum_abs_subdominance_test"
table_decimals = 3
# Print each result's standard deviation over the trials after its mean (if there are several).
table_show_std = True
# Add a row for Fairness Reward Weight Optimization's unadjusted run (equal reward weights).
table_include_unadjusted = True
geomean_column = "Geometric Mean"
table_dir = output_dir("exp_tables", WANDB_CONFIG)


def latex_escape(text):
    """`text` with the characters LaTeX treats specially escaped."""
    special = {
        "\\": r"\textbackslash{}",
        "&": r"\&",
        "%": r"\%",
        "$": r"\$",
        "#": r"\#",
        "_": r"\_",
        "{": r"\{",
        "}": r"\}",
        "~": r"\textasciitilde{}",
        "^": r"\textasciicircum{}",
    }
    return "".join(special.get(char, char) for char in str(text))


def results_table(expert, dataset_bias_type, trial_values, include_unadjusted):
    """A per-trial result of every model on every dataset, against `expert`.

    `trial_values(dataset, algorithm, weight_adjust)` gives one configuration's result per trial,
    as a pandas.Series.

    Returns
    -------
    mean, std : pandas.DataFrame
        The mean and standard deviation over the trials, one row per row of `model_rows()` the
        session ran, one column per dataset; NaN wherever the session did not run a model on a
        dataset (and, for `std`, wherever it ran a single trial).
    """
    rows = model_rows(expert, include_unadjusted)
    datasets = [dataset for dataset in averaged_data if expert in averaged_data[dataset]]
    mean = pd.DataFrame(index=[label for label, *_ in rows], columns=datasets, dtype=float)
    std = mean.copy()
    for label, algorithm, unadjusted in rows:
        for dataset in datasets:
            weight_adjust = row_weight_adjust(
                algorithm, unadjusted, dataset, expert, dataset_bias_type
            )
            if weight_adjust is None:
                continue
            values = trial_values(dataset, algorithm, weight_adjust)
            mean.loc[label, dataset] = values.mean()
            if len(values) > 1:
                std.loc[label, dataset] = values.std(ddof=1)
    ran = mean.notna().any(axis=1)
    return mean[ran], std[ran]


def cell_text(mean, std, decimals, show_std, latex):
    if pd.isna(mean):
        return "--"
    text = f"{mean:.{decimals}f}"
    if show_std and not pd.isna(std):
        text += (r" $\pm$ " if latex else " ± ") + f"{std:.{decimals}f}"
    return text


def latex_table(mean, std, bold, underline, caption, label, decimals, show_std, rule_before=None,
                corner="Model"):
    """`mean` (+- `std`) as a complete LaTeX `table` environment. A rule sets apart the columns
    from `rule_before` on (e.g. a summary column)."""
    column_spec = "".join(
        "|r" if column == rule_before else "r" for column in mean.columns
    )
    lines = [
        r"\begin{table}[htbp]",
        r"\centering",
        rf"\caption{{{caption}}}",
        rf"\label{{{label}}}",
        r"\begin{tabular}{l" + column_spec + "}",
        r"\hline",
        " & ".join([corner] + [latex_escape(column) for column in mean.columns]) + r" \\",
        r"\hline",
    ]
    for row in mean.index:
        cells = []
        for column in mean.columns:
            text = cell_text(mean.loc[row, column], std.loc[row, column], decimals, show_std, True)
            if text != "--" and underline.loc[row, column]:
                text = rf"\underline{{{text}}}"
            if text != "--" and bold.loc[row, column]:
                text = rf"\textbf{{{text}}}"
            cells.append(text)
        lines.append(" & ".join([latex_escape(row)] + cells) + r" \\")
    lines += [r"\hline", r"\end{tabular}", r"\end{table}"]
    return "\n".join(lines) + "\n"


def display_table(mean, std, bold, underline, caption, decimals, show_std):
    text = pd.DataFrame(
        [
            [
                cell_text(mean.loc[row, column], std.loc[row, column], decimals, show_std, False)
                for column in mean.columns
            ]
            for row in mean.index
        ],
        index=mean.index,
        columns=mean.columns,
    )
    styles = pd.DataFrame("", index=mean.index, columns=mean.columns)
    styles = styles.mask(bold.fillna(False).astype(bool), "font-weight: bold;")
    styles = styles + underline.fillna(False).astype(bool).map(
        lambda marked: " text-decoration: underline;" if marked else ""
    )
    display(text.style.set_caption(caption).apply(lambda _: styles, axis=None))


def save_tables(path, latex_tables):
    with open(path, "w") as table_file:
        table_file.write("\n".join(latex_tables))
    print(f"Saved {path}")


def with_dataset_labels(frame):
    """`frame` with its dataset columns named as `dataset_label()` names them."""
    return frame.rename(columns=dataset_label)


experts = table_experts if table_experts is not None else session_experts()
if not experts:
    raise ValueError(f"W&B session {session_id!r} has no runs.")

subdominance_tables = {}
all_latex_tables = []
for expert in experts:
    bias_types = table_bias_types if table_bias_types is not None else session_bias_types(expert)
    summary_columns = {}
    column_names = bias_type_columns(bias_types)
    for table_bias_type in bias_types:
        mean, std = results_table(
            expert,
            table_bias_type,
            lambda dataset, algorithm, weight_adjust: per_trial_data[dataset][expert][algorithm][
                table_bias_type
            ][weight_adjust][table_metric],
            table_include_unadjusted,
        )
        if mean.empty:
            print(f"{expert}: the session ran no model under {table_bias_type!r}; no table made")
            continue
        # Only for a technique with a result on every dataset
        with np.errstate(divide="ignore"):  # a subdominance of 0 makes the mean 0
            mean[geomean_column] = [
                gmean(row) if row.notna().all() else np.nan for _, row in mean.iterrows()
            ]
        std[geomean_column] = np.nan
        mean, std = with_dataset_labels(mean), with_dataset_labels(std)
        subdominance_tables[(expert, table_bias_type)] = (mean, std)
        summary_columns[column_names[table_bias_type]] = mean[geomean_column]

        printed = mean.round(table_decimals)
        # The best (lowest) result of each dataset
        best = printed.rank(method="dense") == 1
        # The results of models that, on average, dominate the expert
        dominates = printed < 1

        bias_text = "" if table_bias_type == () else (
            f" trained on demonstrations with the {latex_escape(bias_type_title(table_bias_type))}"
            " bias, against those biased demonstrations"
        )
        caption = (
            f"Sum of absolute subdominance of each model against the {latex_escape(expert)}"
            f" expert's demonstrations on each dataset's test split{bias_text} (lower is better),"
            f" as the mean over the trials"
            + (r" $\pm$ its standard deviation" if table_show_std else "")
            + f", and the geometric mean of the means over the datasets ({geomean_column})."
            " The best result in each column is in bold, and results below one, where the model on"
            " average dominates the expert, are underlined."
            + (
                " FRWO (equal weights) is FRWO with equal reward weights, before the weight adjustment"
                " learns them from the demonstrations."
                if table_include_unadjusted else ""
            )
        )
        latex = latex_table(
            mean, std, best, dominates, caption,
            f"tab:subdominance_{expert}_{bias_type_label(table_bias_type)}",
            table_decimals, table_show_std, rule_before=geomean_column,
        )
        all_latex_tables.append(latex)
        table_path = os.path.join(
            table_dir, f"subdominance_results_{expert}_{slug(bias_type_label(table_bias_type))}.tex"
        )
        save_tables(table_path, [latex])

        frwo_weight_adjusts = {
            dataset: model_weight_adjust(weight_adjust_algorithm, dataset, expert, table_bias_type)
            for dataset in averaged_data
            if expert in averaged_data[dataset]
        }
        print(
            f"{weight_adjust_algorithm} weight adjustments: "
            + ", ".join(
                f"{dataset}: {weight_adjust_label(weight_adjust)}"
                for dataset, weight_adjust in frwo_weight_adjusts.items()
                if weight_adjust is not None
            )
        )
        display_table(
            mean, std, best, dominates,
            f"Subdominance against the {expert} expert's demonstrations ({bias_type_title(table_bias_type)})",
            table_decimals, table_show_std,
        )

    if not summary_columns:
        continue
    # Every technique's geometric mean over the datasets, under every bias type
    summary = pd.DataFrame(summary_columns)
    printed = summary.round(table_decimals)
    best = printed.rank(method="dense") == 1
    dominates = printed < 1
    no_std = pd.DataFrame(np.nan, index=summary.index, columns=summary.columns)
    caption = (
        f"Geometric mean over the datasets of the sum of absolute subdominance of each model against"
        f" the {latex_escape(expert)} expert's demonstrations on the test split, for models trained"
        " on the unbiased demonstrations and on demonstrations with each bias (each scored against"
        " the demonstrations it was trained on; lower is better). The best result in each column is"
        " in bold, and results below one are underlined."
    )
    latex = latex_table(
        summary, no_std, best, dominates, caption, f"tab:subdominance_summary_{expert}",
        table_decimals, False, corner="Model",
    )
    all_latex_tables.append(latex)
    save_tables(os.path.join(table_dir, f"subdominance_summary_{expert}.tex"), [latex])
    display_table(
        summary, no_std, best, dominates,
        f"Subdominance against the {expert} expert: geometric mean over the datasets",
        table_decimals, False,
    )

# Every table, for `\input{subdominance_results.tex}`
save_tables(os.path.join(table_dir, "subdominance_results.tex"), all_latex_tables)

In [ ]:
# Paired Differences Between FRWO and Each Baseline:

# Every trial gives every technique the same data split, the same demonstrations and, restart by
# restart, the same seeds, so the fair comparison of Fairness Reward Weight Optimization with a
# baseline is trial by trial. One table per expert and bias type: one row per baseline technique,
# one column per dataset, each cell the mean over the trials of FRWO's test subdominance minus the
# baseline's (`paired_metric`, with each dataset's tuned weight adjustment for FRWO), +- the
# half-width of its 95% confidence interval (Student's t over the trials' differences; none with a
# single trial). Negative means FRWO is less subdominant -- better. A difference is bold where its
# whole interval is below 0 (FRWO significantly better), and underlined where it is above 0 (the
# baseline significantly better). The last column is the mean of the datasets' means.
#
# Saved like the subdominance tables, to `paired_differences_<expert>_<bias type>.tex` and, all of
# them, `paired_differences.tex`. Run the subdominance tables cell above first.

paired_metric = "sum_abs_subdominance_test"
paired_decimals = 3
paired_experts = None  # None: every expert the session ran
paired_bias_types = None  # None: every bias type the session ran
paired_mean_column = "Mean"


def paired_differences(expert, dataset_bias_type):
    """FRWO's `paired_metric` minus each baseline's, by trial.

    Returns
    -------
    mean, half_width : pandas.DataFrame
        The mean over the trials and the half-width of its 95% confidence interval, one row per
        baseline technique, one column per dataset.
    """
    datasets = [dataset for dataset in averaged_data if expert in averaged_data[dataset]]
    baselines = [a for a in session_algorithms(expert) if a != weight_adjust_algorithm]
    labels = [algorithm_label(algorithm) for algorithm in baselines]
    mean = pd.DataFrame(np.nan, index=labels, columns=datasets)
    half_width = mean.copy()
    for label, algorithm in zip(labels, baselines):
        for dataset in datasets:
            frwo_weight_adjust = row_weight_adjust(
                weight_adjust_algorithm, False, dataset, expert, dataset_bias_type
            )
            if frwo_weight_adjust is None or dataset_bias_type not in averaged_data[dataset][
                expert
            ].get(algorithm, {}):
                continue
            by_algorithm = per_trial_data[dataset][expert]
            frwo = by_algorithm[weight_adjust_algorithm][dataset_bias_type][frwo_weight_adjust]
            baseline = by_algorithm[algorithm][dataset_bias_type][()]
            # Aligned on TRIAL: only the trials both ran are compared
            differences = (frwo[paired_metric] - baseline[paired_metric]).dropna()
            if differences.empty:
                continue
            mean.loc[label, dataset] = differences.mean()
            if len(differences) > 1:
                half_width.loc[label, dataset] = ci95(differences)
    ran = mean.notna().any(axis=1)
    return mean[ran], half_width[ran]


all_latex_tables = []
for expert in paired_experts if paired_experts is not None else session_experts():
    for dataset_bias_type in (
        paired_bias_types if paired_bias_types is not None else session_bias_types(expert)
    ):
        mean, half_width = paired_differences(expert, dataset_bias_type)
        if mean.empty:
            continue
        mean[paired_mean_column] = [
            row.mean() if row.notna().all() else np.nan for _, row in mean.iterrows()
        ]
        half_width[paired_mean_column] = np.nan
        mean, half_width = with_dataset_labels(mean), with_dataset_labels(half_width)
        # Judged on the values as printed, like the other tables' marks
        upper = (mean + half_width).round(paired_decimals)
        lower = (mean - half_width).round(paired_decimals)
        frwo_better = upper < 0
        baseline_better = lower > 0

        title = bias_type_title(dataset_bias_type)
        caption = (
            f"Difference between the test subdominance of FRWO and of each baseline, against the"
            f" {latex_escape(expert)} expert's demonstrations"
            + ("" if dataset_bias_type == () else f" with the {latex_escape(title)} bias")
            + r", as the mean over the trials of each trial's difference $\pm$ the half-width of"
            " its 95\\% confidence interval, and the mean over the datasets"
            f" ({paired_mean_column}). Negative values favor FRWO. Differences significantly below 0"
            " (FRWO better) are in bold, and those significantly above 0 (the baseline better)"
            " are underlined."
        )
        latex = latex_table(
            mean, half_width, frwo_better, baseline_better, caption,
            f"tab:paired_differences_{expert}_{bias_type_label(dataset_bias_type)}",
            paired_decimals, True, rule_before=paired_mean_column, corner="Baseline",
        )
        all_latex_tables.append(latex)
        save_tables(
            os.path.join(
                table_dir,
                f"paired_differences_{expert}_{slug(bias_type_label(dataset_bias_type))}.tex",
            ),
            [latex],
        )
        display_table(
            mean, half_width, frwo_better, baseline_better,
            f"FRWO − baseline test subdominance, by trial ({expert} expert, {title})",
            paired_decimals, True,
        )

save_tables(os.path.join(table_dir, "paired_differences.tex"), all_latex_tables)

In [ ]:
# Pareto Dominance Rate Tables:

# The share of the expert's demonstrations each model Pareto dominates: is at least as good as on
# every subdominance metric and strictly better on at least one (`dominance_weak` also counts a
# model that ties a demonstration on every metric). This is the direct evidence that a low
# subdominance means the model is mostly Pareto dominant over the expert. Each demonstration is one
# of the expert's subdominance groups on the test split, the same groups subdominance is computed
# over, so that 100% means the model dominates every one of them.
#
# The model is compared with each group on that group's own rows, as subdominance compares them,
# whenever the session recorded the model's losses per group (`model_demo_feat_loss_*`). A session
# logged before those were recorded only has the model's loss on the whole split, which is then
# compared with every group instead (the Superhuman Fairness paper's gamma-superhuman), and the
# table says so. So does a dataset bias type compared with the unbiased expert, whose groups are
# not on the same rows.
#
# Every bias type gets two tables: against the biased demonstrations the model learned from, and
# against the unbiased expert's demonstrations of the same trial (see REFERENCE_TITLE). One row per
# technique (as in the subdominance tables), one column per dataset, as the mean over the trials,
# +- its standard deviation, and a last column with each technique's mean over the datasets. The
# best result in each column is in bold. A summary table per expert and reference then gives each
# technique's mean over the datasets under every bias type.
#
# Saved like the subdominance tables, to `dominance_rate_<expert>_<bias type>[_vs_<reference>].tex`,
# `dominance_summary_<expert>_vs_<reference>.tex` and, all of them, `dominance_rates.tex`.

dominance_split = "test"
dominance_bias_types = None  # None: every bias type the session ran
dominance_experts = None  # None: every expert the session ran
dominance_decimals = 2
dominance_show_std = True
dominance_weak = False
dominance_include_unadjusted = True
mean_column = "Mean"
# Whether each table's comparisons were paired group by group, filled in as they are computed.
dominance_pairing = {}


def dominated_fraction(model_losses, expert_losses, weak=False):
    """The share of the rows of `expert_losses` (groups x metrics) that `model_losses` Pareto
    dominates: row by row if it has the same shape, or one loss per metric compared with every
    row."""
    model_losses = np.asarray(model_losses, dtype=float)
    expert_losses = np.asarray(expert_losses, dtype=float)
    no_worse = (model_losses <= expert_losses).all(axis=1)
    if weak:
        return no_worse.mean()
    return (no_worse & (model_losses < expert_losses).any(axis=1)).mean()


def dominance_rates(dataset, expert, algorithm, dataset_bias_type, weight_adjust, reference,
                    split=None, results=None):
    """The share of the reference expert's demonstrations one configuration dominates, per trial.

    Returns
    -------
    rates : pandas.Series
        Indexed by trial.
    paired : bool
        Whether every trial was compared with the groups row by row.
    """
    results = main_results if results is None else results
    split = dominance_split if split is None else split
    trials = results.per_trial_data[dataset][expert][algorithm][dataset_bias_type][weight_adjust]
    reference_bias_type = dataset_bias_type if reference == "demos" else ()
    # A dataset bias type samples its own groups, so they are not the unbiased run's rows
    same_rows = reference == "demos" or dataset_bias_type == () or is_demo_bias(dataset_bias_type)
    rates = {}
    paired = True
    for trial in trials.index.unique():
        expert_losses = get_expert_demo_losses(
            dataset, expert, reference_bias_type, trial=trial, split=split, results=results
        )
        model_losses = get_model_demo_losses(
            dataset, expert, algorithm, dataset_bias_type, weight_adjust, trial=trial,
            split=split, results=results,
        )
        if model_losses is not None and same_rows:
            model = model_losses[expert_losses.columns].to_numpy()
        else:
            paired = False
            trial_results = trials.loc[[trial]].iloc[0]
            model = feature_losses_of(trial_results, expert_losses.columns, split)
        rates[trial] = dominated_fraction(model, expert_losses.to_numpy(), dominance_weak)
    return pd.Series(rates, dtype=float), paired


def feature_losses_of(results, metrics, split):
    """A model's loss on each of `metrics` on the whole split, from one trial's results."""
    losses = []
    for metric in metrics:
        if f"model_feat_loss_{split}_{metric}" in results.index:
            losses.append(results[f"model_feat_loss_{split}_{metric}"])
        else:
            # Sessions logged before model_feat_loss_* was recorded: the feature expectation is a
            # "goodness" measure, so its loss is 1 - mu
            losses.append(1 - results[f"muL_{split}_{metric}"])
    return np.asarray(losses, dtype=float)


experts = dominance_experts if dominance_experts is not None else session_experts()
dominance_tables = {}
all_latex_tables = []
for expert in experts:
    bias_types = (
        dominance_bias_types if dominance_bias_types is not None else session_bias_types(expert)
    )
    summary_columns = {"demos": {}, "unbiased": {}}
    column_names = bias_type_columns(bias_types)
    for dominance_bias_type in bias_types:
        for reference in references_for(dominance_bias_type):
            pairing = set()

            def trial_rates(dataset, algorithm, weight_adjust):
                rates, paired = dominance_rates(
                    dataset, expert, algorithm, dominance_bias_type, weight_adjust, reference
                )
                pairing.add(paired)
                return rates

            mean, std = results_table(
                expert, dominance_bias_type, trial_rates, dominance_include_unadjusted
            )
            if mean.empty:
                continue
            mean[mean_column] = [
                row.mean() if row.notna().all() else np.nan for _, row in mean.iterrows()
            ]
            std[mean_column] = np.nan
            mean, std = with_dataset_labels(mean), with_dataset_labels(std)
            dominance_tables[(expert, dominance_bias_type, reference)] = (mean, std)
            dominance_pairing[(expert, dominance_bias_type, reference)] = pairing == {True}
            title = bias_type_title(dominance_bias_type)
            summary_columns[reference][column_names[dominance_bias_type]] = mean[mean_column]
            if dominance_bias_type == ():
                # Under the unbiased dataset, the two references are the same expert
                summary_columns["unbiased"][column_names[dominance_bias_type]] = mean[mean_column]

            best = mean.round(dominance_decimals).rank(method="dense", ascending=False) == 1
            no_marks = pd.DataFrame(False, index=mean.index, columns=mean.columns)
            pairing_text = (
                " Each model is compared with each demonstration on that demonstration's rows."
                if pairing == {True}
                else " Each model's loss on the whole test split is compared with every"
                " demonstration."
            )
            if dominance_bias_type == ():
                against = f"the {latex_escape(expert)} expert's demonstrations"
            elif reference == "demos":
                against = (
                    f"the {latex_escape(expert)} expert's demonstrations with the"
                    f" {latex_escape(title)} bias, which the models were trained on"
                )
            else:
                against = (
                    f"the {latex_escape(expert)} expert's unbiased demonstrations, for models"
                    f" trained on demonstrations with the {latex_escape(title)} bias"
                )
            caption = (
                f"Share of {against} that each model Pareto dominates on each dataset's test split"
                f" (higher is better), as the mean over the trials"
                + (r" $\pm$ its standard deviation" if dominance_show_std else "")
                + f", and its mean over the datasets ({mean_column}). A model dominates a"
                " demonstration when it is no worse on every subdominance metric"
                + ("" if dominance_weak else " and better on at least one")
                + "." + pairing_text + " The best result in each column is in bold."
            )
            suffix = reference_suffix(dominance_bias_type, reference)
            latex = latex_table(
                mean, std, best, no_marks, caption,
                f"tab:dominance_{expert}_{bias_type_label(dominance_bias_type)}{suffix}",
                dominance_decimals, dominance_show_std, rule_before=mean_column,
            )
            all_latex_tables.append(latex)
            save_tables(
                os.path.join(
                    table_dir,
                    f"dominance_rate_{expert}_{slug(bias_type_label(dominance_bias_type))}{suffix}.tex",
                ),
                [latex],
            )
            if pairing != {True}:
                print("Not paired group by group: compared the model's loss on the whole split.")
            display_table(
                mean, std, best, no_marks,
                f"Share of the {expert} expert's demonstrations dominated ({title},"
                f" vs. {REFERENCE_TITLE[reference]})",
                dominance_decimals, dominance_show_std,
            )

    for reference, columns in summary_columns.items():
        if not columns or (reference == "unbiased" and len(columns) == 1):
            continue
        summary = pd.DataFrame(columns)
        best = summary.round(dominance_decimals).rank(method="dense", ascending=False) == 1
        no_marks = pd.DataFrame(False, index=summary.index, columns=summary.columns)
        no_std = pd.DataFrame(np.nan, index=summary.index, columns=summary.columns)
        caption = (
            f"Mean over the datasets of the share of the {latex_escape(expert)} expert's"
            " demonstrations each model Pareto dominates on the test split, for models trained on"
            " the unbiased demonstrations and on demonstrations with each bias, compared with "
            + (
                "the demonstrations each was trained on"
                if reference == "demos"
                else "the expert's unbiased demonstrations"
            )
            + " (higher is better). The best result in each column is in bold."
        )
        latex = latex_table(
            summary, no_std, best, no_marks, caption,
            f"tab:dominance_summary_{expert}_vs_{reference}", dominance_decimals, False,
        )
        all_latex_tables.append(latex)
        save_tables(
            os.path.join(table_dir, f"dominance_summary_{expert}_vs_{reference}.tex"), [latex]
        )
        display_table(
            summary, no_std, best, no_marks,
            f"Share of the {expert} expert's demonstrations dominated: mean over the datasets"
            f" (vs. {REFERENCE_TITLE[reference]})",
            dominance_decimals, False,
        )

save_tables(os.path.join(table_dir, "dominance_rates.tex"), all_latex_tables)

In [ ]:
# Shared setup for the Pareto scatter plots below, which grew out of `plot_features` and
# `plot_features_errorbars` of `plot.py` of the Superhuman Fairness reference implementation
# (https://github.com/omidMemari/superhumn-fairness).
#
# They scatter, for pairs of performance/fairness losses, the expert's demonstrations -- one point
# per subdominance group -- against every model the session ran, each at its mean over the
# session's trials with a 95% confidence interval. Every value is a loss: lower is better, so a
# model below and to the left of a demonstration dominates it on that pair.
#
# They differ from the originals in these ways:
# * The losses are this project's Acc (as prediction error) and the differences of AccPar, DemPar,
#   EqOpp and TNRPar, in place of the paper's error, DP, EqOdds and PRP.
# * Nothing specific to Superhuman Fairness is drawn: no 1/alpha subdominance margins, and no
#   Superhuman Fairness training-split point. Every model, Superhuman Fairness among them, is drawn
#   the same way, on the same split.
# * Every technique the session ran is drawn, the Superhuman Fairness variants among them, and a
#   figure no longer needs the session to have run Superhuman Fairness.
# * The demonstrations are those of the same split as the models (`pareto_split`), rather than of
#   the training split.
# * Under a demo bias type, both the biased demonstrations the models learned from and the
#   expert's unbiased demonstrations are drawn.
# * The fair log-loss models are drawn at their measured losses, like every other model, rather
#   than at their own expected error and fairness violation.
# * The gamma-superhuman printouts are replaced by the Pareto dominance rate tables above.

# The losses to plot, in order. Each must be one of the SUBDOMINANCE_*_METRICS_LIST the session
# ran with, since those are the measures both the expert's demonstrations and the models are
# recorded on. Those may be named as this project's objectives ("Acc", "DemPar", ...) or as the
# original paper's metrics ("inacc", "dp", "prp", ...). None plots every metric of the session, in
# order. The first one is the performance loss every fairness loss is plotted against.
feature_list = None

# The metrics the loaded session actually recorded, in its own order.
session_metric_names = list(
    next(iter(expert_demo_losses.values()))["expert_demo_feat_loss_metrics"]
)
if feature_list is None:
    feature_list = session_metric_names
missing_metrics = [f for f in feature_list if f not in session_metric_names]
if missing_metrics:
    raise ValueError(
        f"feature_list names {missing_metrics}, which W&B session {session_id!r}"
        f" did not record. It ran with {session_metric_names}; plot a subset of"
        " those, or re-run the training script with the metrics you want in"
        " SUBDOMINANCE_PERF_METRICS_LIST / SUBDOMINANCE_FAIR_METRICS_LIST."
    )

# Axis labels, in the style of the original's `name` and `short` dicts. A
# metric with no entry falls back to its own name.
name = {
    "Acc": "Prediction error",
    "AccPar": "D.AccPar",
    "DemPar": "D.DP",
    "EqOpp": "D.EqOpp",
    "TNRPar": "D.TNR",
    "FPRPar": "D.FPR",
    "FNRPar": "D.FNR",
    "EqOdds": "D.EqOdds",
    "PredPar": "D.PPV",
    "NegPredPar": "D.NPV",
    "inacc": "Prediction error",
    "dp": "D.DP",
    "eqodds": "D.EqOdds",
    "prp": "D.PRP",
    "eqopp": "D.FNR",
    "fnr": "D.FNR",
    "fpr": "D.FPR",
    "ppv": "D.PPV",
    "npv": "D.NPV",
    "error_rate_diff": "D.Balanced Error Rate",
}
short = {
    "Acc": "error",
    "AccPar": "AccPar",
    "DemPar": "DP",
    "EqOpp": "EqOpp",
    "TNRPar": "TNR",
    "FPRPar": "FPR",
    "FNRPar": "FNR",
    "EqOdds": "EqOdds",
    "PredPar": "PPV",
    "NegPredPar": "NPV",
    "inacc": "error",
    "dp": "DP",
    "eqodds": "EqOdds",
    "prp": "PRP",
    "eqopp": "FNR",
    "fnr": "FNR",
    "fpr": "FPR",
    "ppv": "PPV",
    "npv": "NPV",
    "error_rate_diff": "D.ErrorRate",
}
name = {**{f: f for f in feature_list}, **name}
short = {**{f: f for f in feature_list}, **short}


def feature_losses(results, split):
    """A model's loss on every plotted feature, from its W&B results.

    `results` is one configuration's entry of `averaged_data` (a Series) or of
    `per_trial_data` (a DataFrame, giving one loss per trial).

    `model_feat_loss_*` holds the model's loss on each subdominance metric
    directly, whichever vocabulary names it. Sessions logged before those were
    recorded fall back to the feature expectations, which are "goodness"
    measures, so a loss is `1 - mu` -- the same inversion the expert's
    demonstration losses were recorded with.
    """
    available = results.index if isinstance(results, pd.Series) else results.columns
    losses = {}
    for f in feature_list:
        if f"model_feat_loss_{split}_{f}" in available:
            losses[f] = results[f"model_feat_loss_{split}_{f}"]
            continue
        for key in (f"muL_perf_{split}_{f}", f"muL_{split}_{f}"):
            if key in available:
                losses[f] = 1 - results[key]
                break
        else:
            raise KeyError(
                f"No {split} loss for {f!r} in the results. Sessions logged"
                " before model_feat_loss_* was recorded only cover metrics named"
                " as this project's objectives; re-run the training script to"
                " plot the original paper's metrics."
            )
    return losses


def resolve_frwo_weight_adjust(dataset, expert, dataset_bias_type):
    """The Fairness Reward Weight Optimization weight adjustment to plot, or None when the
    session did not run Fairness Reward Weight Optimization for this configuration (the
    figures then leave it out)."""
    frwo_results = averaged_data[dataset][expert].get(weight_adjust_algorithm, {})
    if dataset_bias_type not in frwo_results:
        return None
    if dataset not in weight_adjust_override and all(
        weight_adjust == () for weight_adjust in frwo_results[dataset_bias_type]
    ):
        return ()
    return tuned_weight_adjust(dataset, expert, dataset_bias_type)

In [ ]:
# Pareto Scatter Plots: run the setup cell above first.
#
# Two kinds of figure, for every expert and bias type of the session:
# * `pareto_<expert>_<bias type>.pdf`: one row per dataset, one column per fairness loss, each
#   panel the performance loss (prediction error) against that fairness loss. The compact figure
#   for the main text.
# * With `pareto_all_pairs`, `pareto_pairs_<dataset>_<expert>_<bias type>.pdf`: every pair of
#   losses of one dataset, for the appendix. These are only saved, not drawn here.
#
# The demonstrations are those of trial `pareto_demo_trial` (each trial draws its own data split,
# so one trial's demonstrations are shown, as the original did), while every model is drawn at its
# mean over the session's trials, with a 95% confidence interval as its error bars (none for a
# single trial). Under a bias type, the biased demonstrations the models learned from are drawn
# as filled points and the expert's unbiased demonstrations as rings.

pareto_split = "test"
pareto_demo_trial = 0
# The techniques to draw. None draws every one the session ran, Fairness Reward Weight
# Optimization first.
pareto_algorithms = None
pareto_experts = None  # None: every expert the session ran
pareto_bias_types = None  # None: every bias type the session ran
pareto_all_pairs = True
pareto_dir = output_dir("exp_plots", WANDB_CONFIG, "pareto")


def pareto_models(dataset, expert, dataset_bias_type):
    """`(algorithm, weight_adjust, per-trial losses)` of every model drawn for one configuration."""
    models = []
    if expert not in averaged_data[dataset]:
        return models
    algorithms = pareto_algorithms if pareto_algorithms is not None else session_algorithms(expert)
    for algorithm in algorithms:
        if algorithm == weight_adjust_algorithm:
            weight_adjust = resolve_frwo_weight_adjust(dataset, expert, dataset_bias_type)
        elif dataset_bias_type in averaged_data[dataset].get(expert, {}).get(algorithm, {}):
            weight_adjust = ()
        else:
            weight_adjust = None
        if weight_adjust is None:
            continue
        trials = per_trial_data[dataset][expert][algorithm][dataset_bias_type][weight_adjust]
        models.append((algorithm, weight_adjust, feature_losses(trials, pareto_split)))
    return models


def demo_trial_of(dataset, expert, dataset_bias_type):
    """`pareto_demo_trial`, or the first trial the session recorded demonstrations for."""
    trials = sorted(
        trial
        for (d, e, b, trial) in expert_demo_losses
        if (d, e, b) == (dataset, expert, dataset_bias_type)
    )
    if not trials:
        return None
    return pareto_demo_trial if pareto_demo_trial in trials else trials[0]


def pareto_demos(dataset, expert, dataset_bias_type):
    """`(label, losses, filled)` of each set of demonstrations drawn for one configuration."""
    trial = demo_trial_of(dataset, expert, dataset_bias_type)
    if trial is None:
        return []
    demos = [(
        "Expert demonstrations" if dataset_bias_type == () else "Biased demonstrations (learned from)",
        get_expert_demo_losses(dataset, expert, dataset_bias_type, trial=trial, split=pareto_split),
        True,
    )]
    if dataset_bias_type != () and (dataset, expert, (), trial) in expert_demo_losses:
        demos.append((
            "Unbiased expert demonstrations",
            get_expert_demo_losses(dataset, expert, (), trial=trial, split=pareto_split),
            False,
        ))
    return demos


def draw_pareto_panel(ax, demos, models, x_feature, y_feature):
    for label, losses, filled in demos:
        if filled:
            ax.scatter(losses[x_feature], losses[y_feature], s=9, color=EXPERT_COLOR, alpha=0.55,
                       linewidths=0, label=label, zorder=2)
        else:
            ax.scatter(losses[x_feature], losses[y_feature], s=12, facecolors="none",
                       edgecolors=INK_SECONDARY, linewidths=0.6, alpha=0.6, label=label, zorder=2)
    for algorithm, weight_adjust, losses in models:
        x, y = losses[x_feature], losses[y_feature]
        is_frwo = algorithm == weight_adjust_algorithm
        ax.errorbar(
            x.mean(), y.mean(), xerr=ci95(x), yerr=ci95(y), fmt=algorithm_marker(algorithm),
            color=algorithm_color(algorithm), markersize=7 if is_frwo else 5,
            markeredgecolor=SURFACE, markeredgewidth=0.8, elinewidth=0.8, capsize=0,
            label=algorithm_label(algorithm), zorder=5 if is_frwo else 4,
        )
    ax.set_xlabel(name[x_feature])
    ax.set_ylabel(name[y_feature])
    ax.margins(0.08)


def pareto_title(expert, dataset_bias_type):
    return f"{expert} expert, {bias_type_title(dataset_bias_type)} (losses: lower is better)"


def plot_pareto_grid(expert, dataset_bias_type):
    """Prediction error against every fairness loss, one row per dataset."""
    datasets = [
        dataset for dataset in averaged_data if pareto_models(dataset, expert, dataset_bias_type)
    ]
    perf_feature, fair_features = feature_list[0], feature_list[1:]
    fig, axes = plt.subplots(
        len(datasets), len(fair_features),
        figsize=(1.9 * len(fair_features), 1.75 * len(datasets) + 0.4), squeeze=False,
    )
    for row, dataset in enumerate(datasets):
        demos = pareto_demos(dataset, expert, dataset_bias_type)
        models = pareto_models(dataset, expert, dataset_bias_type)
        for column, fair_feature in enumerate(fair_features):
            ax = axes[row, column]
            draw_pareto_panel(ax, demos, models, fair_feature, perf_feature)
            if column:
                ax.set_ylabel("")
            else:
                ax.set_ylabel(f"{dataset_label(dataset)}\n{name[perf_feature]}")
            if row < len(datasets) - 1:
                ax.set_xlabel("")
    figure_title(fig, pareto_title(expert, dataset_bias_type))
    fig.tight_layout()
    shared_legend(fig, axes)
    save_figure(fig, os.path.join(
        pareto_dir, f"pareto_{expert}_{slug(bias_type_label(dataset_bias_type))}.pdf"
    ))


def plot_pareto_pairs(dataset, expert, dataset_bias_type):
    """Every pair of losses of one dataset."""
    pairs = list(itertools.combinations(feature_list, 2))
    n_columns = min(len(pairs), 5)
    n_rows = int(np.ceil(len(pairs) / n_columns))
    fig, axes = plt.subplots(n_rows, n_columns, figsize=(1.9 * n_columns, 1.8 * n_rows + 0.4),
                             squeeze=False)
    demos = pareto_demos(dataset, expert, dataset_bias_type)
    models = pareto_models(dataset, expert, dataset_bias_type)
    for ax, (y_feature, x_feature) in zip(axes.flat, pairs):
        draw_pareto_panel(ax, demos, models, x_feature, y_feature)
    for ax in axes.flat[len(pairs):]:
        ax.set_visible(False)
    figure_title(fig, f"{dataset_label(dataset)}: {pareto_title(expert, dataset_bias_type)}")
    fig.tight_layout()
    shared_legend(fig, axes)
    save_figure(fig, os.path.join(
        pareto_dir,
        f"pareto_pairs_{dataset}_{expert}_{slug(bias_type_label(dataset_bias_type))}.pdf",
    ), show=False)


for expert in pareto_experts if pareto_experts is not None else session_experts():
    for dataset_bias_type in (
        pareto_bias_types if pareto_bias_types is not None else session_bias_types(expert)
    ):
        plot_pareto_grid(expert, dataset_bias_type)
        if pareto_all_pairs:
            for dataset in averaged_data:
                if expert in averaged_data[dataset] and pareto_models(dataset, expert, dataset_bias_type):
                    plot_pareto_pairs(dataset, expert, dataset_bias_type)

In [ ]:
# Objective Differences Between Each Model and the Expert:

# One heatmap per expert, bias type and reference expert (see REFERENCE_TITLE): one row per model
# (as in the subdominance tables), one column per dataset and objective, each cell the model's
# test feature expectation minus the reference expert's, in percentage points, as the mean over the
# trials of each trial's difference. Every objective is a "goodness" measure, so blue is where the
# model is better than the expert and red where it is worse. Each row's label ends with the number
# of its cells where the model is at least as good as the expert, so that "dominates the expert on
# most objectives for most datasets" reads as a mostly blue row with a count near its total.
#
# The color scale is bounded by the `heatmap_vmax_percentile` percentile of the absolute
# differences, so that one outlier (e.g. a collapsed baseline) does not wash out the rest; larger
# differences take the scale's end color, and the colorbar's pointed ends mark that.

heatmap_split = "test"
heatmap_experts = None  # None: every expert the session ran
heatmap_bias_types = None  # None: every bias type the session ran
heatmap_include_unadjusted = True
heatmap_annotate = False  # write each cell's value in it
heatmap_vmax_percentile = 95
heatmap_dir = output_dir("exp_plots", WANDB_CONFIG, "objective_differences")


def objective_differences(expert, dataset_bias_type, reference):
    """Each model's mean difference from the reference expert, per dataset and objective.

    Returns
    -------
    differences : pandas.DataFrame
        One row per row of `model_rows()`, one column per (dataset, objective).
    """
    rows = model_rows(expert, heatmap_include_unadjusted)
    datasets = [dataset for dataset in averaged_data if expert in averaged_data[dataset]]
    columns = pd.MultiIndex.from_product([datasets, measurable_metrics])
    differences = pd.DataFrame(np.nan, index=[label for label, *_ in rows], columns=columns)
    for label, algorithm, unadjusted in rows:
        for dataset in datasets:
            weight_adjust = row_weight_adjust(algorithm, unadjusted, dataset, expert, dataset_bias_type)
            if weight_adjust is None:
                continue
            trials = per_trial_data[dataset][expert][algorithm][dataset_bias_type][weight_adjust]
            for metric in measurable_metrics:
                differences.loc[label, (dataset, metric)] = (
                    trials[f"muL_{heatmap_split}_{metric}"]
                    - trials[expert_mu_key(heatmap_split, metric, reference)]
                ).mean()
    return differences.dropna(how="all")


def plot_objective_differences(expert, dataset_bias_type, reference):
    differences = objective_differences(expert, dataset_bias_type, reference)
    if differences.empty:
        return
    values = differences.to_numpy() * 100
    at_least_as_good = (differences >= 0).sum(axis=1)
    measured = differences.notna().sum(axis=1)
    row_labels = [
        f"{label}  {at_least_as_good[label]}/{measured[label]}" for label in differences.index
    ]
    datasets = list(dict.fromkeys(differences.columns.get_level_values(0)))
    n_metrics = len(measurable_metrics)

    fig, ax = plt.subplots(
        figsize=(0.32 * values.shape[1] + 2.2, 0.3 * values.shape[0] + 1.4)
    )
    vmax = np.nanpercentile(np.abs(values), heatmap_vmax_percentile) if np.isfinite(values).any() else 1
    image = diverging_heatmap(
        ax, values, row_labels, list(differences.columns.get_level_values(1)), vmax=vmax,
        annotate=heatmap_annotate,
    )
    plt.setp(ax.get_xticklabels(), rotation=90)
    # The datasets along the top, with a gap between them
    top = ax.secondary_xaxis("top")
    top.set_xticks([i * n_metrics + (n_metrics - 1) / 2 for i in range(len(datasets))])
    top.set_xticklabels([dataset_label(dataset) for dataset in datasets])
    top.tick_params(length=0)
    top.spines["top"].set_visible(False)
    for i in range(1, len(datasets)):
        ax.axvline(i * n_metrics - 0.5, color=SURFACE, linewidth=3)
    colorbar_beside(fig, ax, image, "Model − expert (percentage points)", extend="both")
    ax.set_title(
        f"{expert} expert, {bias_type_title(dataset_bias_type)}: vs. {REFERENCE_TITLE[reference]}",
        pad=18,
    )
    fig.tight_layout()
    save_figure(fig, os.path.join(
        heatmap_dir,
        f"objective_differences_{expert}_{slug(bias_type_label(dataset_bias_type))}"
        f"{reference_suffix(dataset_bias_type, reference)}.pdf",
    ))


for expert in heatmap_experts if heatmap_experts is not None else session_experts():
    for dataset_bias_type in (
        heatmap_bias_types if heatmap_bias_types is not None else session_bias_types(expert)
    ):
        for reference in references_for(dataset_bias_type):
            plot_objective_differences(expert, dataset_bias_type, reference)

In [ ]:
# Pareto Dominance Under Each Demonstration Bias: run the Pareto dominance rate cell first.

# Whether Fairness Reward Weight Optimization dominates a low-quality expert: the demo bias types
# turn the expert into one, so one figure per expert shows, for every bias type (the unbiased
# dataset first, as the baseline), the share of the expert's test demonstrations Fairness Reward
# Weight Optimization dominates, as the mean over the trials with a 95% confidence interval. One
# column per dataset; the top row compares it with the biased demonstrations it learned from and
# the bottom row with the expert's unbiased demonstrations (see REFERENCE_TITLE). The diamond is the
# best result of any baseline technique on the same comparison, for context; the dominance rate
# tables above give each one's.

bias_dominance_experts = None  # None: every expert the session ran
bias_dominance_dir = output_dir("exp_plots", WANDB_CONFIG, "bias_dominance")


def configuration_rate(dataset, expert, algorithm, dataset_bias_type, reference):
    """The mean and 95% confidence interval of one technique's dominance rate, or None if the
    session did not run it (Fairness Reward Weight Optimization with its tuned weights)."""
    by_bias_type = averaged_data[dataset].get(expert, {}).get(algorithm, {})
    if dataset_bias_type not in by_bias_type:
        return None
    weight_adjust = row_weight_adjust(algorithm, False, dataset, expert, dataset_bias_type)
    rates, _paired = dominance_rates(
        dataset, expert, algorithm, dataset_bias_type, weight_adjust, reference
    )
    return rates.mean(), ci95(rates)


def plot_bias_dominance(expert):
    bias_types = session_bias_types(expert)
    datasets = [dataset for dataset in averaged_data if expert in averaged_data[dataset]]
    baselines = [a for a in session_algorithms(expert) if a != weight_adjust_algorithm]
    fig, axes = plt.subplots(2, len(datasets), figsize=(2.0 * len(datasets) + 0.6, 4.3),
                             squeeze=False, sharey=True)
    x = np.arange(len(bias_types))
    for column, dataset in enumerate(datasets):
        for row, reference in enumerate(("demos", "unbiased")):
            ax = axes[row, column]
            frwo = [
                configuration_rate(dataset, expert, weight_adjust_algorithm, bias_type, reference)
                for bias_type in bias_types
            ]
            means = [np.nan if r is None else r[0] for r in frwo]
            errors = [0 if r is None else r[1] for r in frwo]
            ax.bar(x, means, 0.6, color=FRWO_COLOR, edgecolor=SURFACE, linewidth=1.0, zorder=2,
                   label=algorithm_label(weight_adjust_algorithm))
            if max(errors) > 0:
                # A share lies in [0, 1], so its interval is cut off there
                means_array, errors_array = np.asarray(means), np.asarray(errors)
                ax.errorbar(x, means, fmt="none", ecolor=INK_SECONDARY, elinewidth=0.8,
                            capsize=1.5, zorder=3, yerr=[
                                np.minimum(errors_array, means_array),
                                np.minimum(errors_array, 1 - means_array),
                            ])
            best_baseline = []
            for bias_type in bias_types:
                rates = [
                    configuration_rate(dataset, expert, baseline, bias_type, reference)
                    for baseline in baselines
                ]
                rates = [r[0] for r in rates if r is not None]
                best_baseline.append(max(rates) if rates else np.nan)
            if baselines:
                ax.scatter(x, best_baseline, marker="D", s=22, facecolors=SURFACE,
                           edgecolors=INK_SECONDARY, linewidths=1.0, zorder=4,
                           label="Best baseline")
            ax.set_xticks(x)
            ax.set_xticklabels([bias_type_short(b) for b in bias_types], rotation=35, ha="right")
            ax.grid(axis="x", visible=False)
            ax.set_ylim(0, 1.02)
            ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
            if row == 0:
                ax.set_title(dataset_label(dataset))
                ax.set_xticklabels([])
    axes[0, 0].set_ylabel("Demos dominated\nvs. demos learned from")
    axes[1, 0].set_ylabel("Demos dominated\nvs. unbiased expert")
    figure_title(fig, f"Share of the {expert} expert's demonstrations dominated under each bias")
    fig.tight_layout()
    shared_legend(fig, axes, order=[algorithm_label(weight_adjust_algorithm), "Best baseline"])
    save_figure(fig, os.path.join(bias_dominance_dir, f"bias_dominance_{expert}.pdf"))


for expert in bias_dominance_experts if bias_dominance_experts is not None else session_experts():
    plot_bias_dominance(expert)

In [ ]:
# Feature Expectations of Each Model and the Expert:

# One figure per technique, expert, bias type and reference expert (see REFERENCE_TITLE): one
# panel per dataset, comparing the expert's test feature expectation of each objective (gray) with
# the model's, side by side, each as the mean over the trials with a 95% confidence interval. The
# number above each pair is the model minus the expert, in percentage points (positive: the model
# is better). Under the unbiased dataset there is one reference; under a bias type, one figure
# compares the model with the biased demonstrations it learned from and another with the expert's
# unbiased demonstrations, which is what the figure of this section used to compare against.
#
# Each dataset's weight adjustment optimizer is tuned separately, so each dataset's Fairness Reward
# Weight Optimization bars are drawn for whichever weight adjustment its own runs used (see
# `tuned_weight_adjust()` in the selection cell above), named in its panel's title. The baselines
# have no reward weights to adjust, so theirs are those of their only, unadjusted `()`, runs.
#
# Saved to `feature_expectations/feat_exp_<technique>_<expert>_<bias type>[_vs_<reference>].pdf`;
# only the techniques of `fe_show_algorithms` are also drawn here, since there are many.

fe_experts = None  # None: every expert the session ran
fe_bias_types = None  # None: every bias type the session ran
# The techniques to draw a figure for. None draws one for every technique the session ran for
# each expert, Fairness Reward Weight Optimization first. Names are those of `selected_algorithm`.
fe_algorithms = None
# fe_algorithms = ["Fairness Reward Weight Optimization", "Superhuman Fairness", "Post Proc DP"]
fe_show_algorithms = [weight_adjust_algorithm]
fe_split = "test"
fe_dir = output_dir("exp_plots", WANDB_CONFIG, "feature_expectations")


def feature_expectations(algorithm, expert, dataset_bias_type, reference):
    """The model's and the reference expert's per-trial feature expectations, per dataset.

    Returns
    -------
    by_dataset : dict
        For each dataset the session ran `algorithm` on under `dataset_bias_type`, a
        `(weight_adjust, model, expert)` triple, `model` and `expert` holding one row per trial and
        one column per metric of `measurable_metrics`.
    """
    by_dataset = {}
    for dataset in averaged_data:
        if expert not in averaged_data[dataset]:
            continue
        weight_adjust = model_weight_adjust(algorithm, dataset, expert, dataset_bias_type)
        if weight_adjust is None:
            continue
        trials = per_trial_data[dataset][expert][algorithm][dataset_bias_type][weight_adjust]
        model = trials[[f"muL_{fe_split}_{metric}" for metric in measurable_metrics]]
        model.columns = measurable_metrics
        expert_fe = trials[[expert_mu_key(fe_split, metric, reference) for metric in measurable_metrics]]
        expert_fe.columns = measurable_metrics
        by_dataset[dataset] = (weight_adjust, model, expert_fe)
    return by_dataset


def plot_feature_expectations(algorithm, expert, dataset_bias_type, reference, by_dataset):
    fig, axes = plt.subplots(1, len(by_dataset), figsize=(2.6 * len(by_dataset) + 0.4, 2.6),
                             squeeze=False, sharey=True)
    for ax, (dataset, (weight_adjust, model, expert_fe)) in zip(axes[0], by_dataset.items()):
        series = [
            (expert_label(dataset_bias_type, reference), expert_fe.mean(),
             expert_fe.apply(ci95), EXPERT_COLOR),
            (algorithm_label(algorithm), model.mean(), model.apply(ci95), algorithm_color(algorithm)),
        ]
        positions = grouped_bars(ax, measurable_metrics, series)
        tops = np.maximum(
            (expert_fe.mean() + expert_fe.apply(ci95)).to_numpy(),
            (model.mean() + model.apply(ci95)).to_numpy(),
        )
        annotate_differences(ax, np.mean(positions, axis=0), tops,
                             (model - expert_fe).mean().to_numpy())
        title = dataset_label(dataset)
        if algorithm == weight_adjust_algorithm:
            title += f"\n{weight_adjust_label(weight_adjust)}"
        ax.set_title(title)
        ax.set_ylim(0, 1.1)
        plt.setp(ax.get_xticklabels(), rotation=35, ha="right")
    axes[0, 0].set_ylabel(f"Feature expectation ({fe_split})")
    figure_title(fig, 
        f"{algorithm_label(algorithm)} vs. {REFERENCE_TITLE[reference]}: {expert} expert,"
        f" {bias_type_title(dataset_bias_type)} (model − expert in pp above each pair)"
    )
    fig.tight_layout()
    shared_legend(fig, axes)
    save_figure(
        fig,
        os.path.join(
            fe_dir,
            f"feat_exp_{slug(algorithm_label(algorithm))}_{expert}_"
            f"{slug(bias_type_label(dataset_bias_type))}"
            f"{reference_suffix(dataset_bias_type, reference)}.pdf",
        ),
        show=SHOW_FIGURES and algorithm in fe_show_algorithms,
    )


for expert in fe_experts if fe_experts is not None else session_experts():
    algorithms = fe_algorithms if fe_algorithms is not None else session_algorithms(expert)
    for dataset_bias_type in fe_bias_types if fe_bias_types is not None else session_bias_types(expert):
        for algorithm in algorithms:
            for reference in references_for(dataset_bias_type):
                by_dataset = feature_expectations(algorithm, expert, dataset_bias_type, reference)
                if not by_dataset:
                    print(
                        f"{algorithm}: the session did not run it for {expert} under"
                        f" {dataset_bias_type!r}; no figure drawn"
                    )
                    continue
                plot_feature_expectations(algorithm, expert, dataset_bias_type, reference, by_dataset)

In [ ]:
# Visualize what happens to the feature expectations throughout the training process if you fully zero the negative weights for each dataset. Show that if you just zero the negative weights, you get something where the feature expectations dont match what the expert originally was attempting
# feature expectations of the dataset with no bias
# feature expectations of the dataset with bias added
# feature expectations of FRWO trained on bias with no zeroing
# Feature expectations of FRWO trained on bias with zeroing
#
# Each dataset is plotted with the weight adjustment its own runs used (see
# `tuned_weight_adjust()` in the selection cell above), and always for Fairness Reward Weight
# Optimization, the only technique that adjusts its weights, whatever `selected_algorithm` is.

# selected_expert = "OptClfMDPPol"
unbiased_type = ()
# biased_type = ("dataset_unbalanced_redlining", 0.2)
# biased_type = ("dataset_balanced_redlining", 0.2)
# biased_type = ("dataset_perfectly_balanced_redlining", 0.2)
# biased_type = ("dataset_corruption_bias", "CatBoost", 0.01, "gaussian", 1.0)
# biased_type = ("demo_balanced_redlining", 0.2)
# biased_type = ("demo_corruption_bias", "CatBoost", 0.01, "gaussian", 1.0)
biased_type = ("demo_noise_bias", 0.2)

muE_test_unbiased_names = ["muE_test_unbiased_" + metric + "_mean" for metric in measurable_metrics]
muE_test_names = ["muE_test_" + metric + "_mean" for metric in measurable_metrics]
muL_test_names = ["muL_test_" + metric for metric in measurable_metrics]

dataset_results = {}

# Every dataset of the loaded session, in the order it was run
for dataset in averaged_data:
    by_bias_type = averaged_data[dataset][selected_expert][weight_adjust_algorithm]
    weight_adjust = tuned_weight_adjust(dataset, selected_expert, biased_type)

    muE_test_unbiased = by_bias_type[unbiased_type][unadjusted_weights][muE_test_unbiased_names].values.tolist()

    muE_test = by_bias_type[biased_type][unadjusted_weights][muE_test_names].values.tolist()

    muL_test_unbiased = by_bias_type[unbiased_type][unadjusted_weights][muL_test_names].values.tolist()

    muL_test_unadjusted = by_bias_type[biased_type][unadjusted_weights][muL_test_names].values.tolist()

    muL_test_adjusted = by_bias_type[biased_type][weight_adjust][muL_test_names].values.tolist()

    dataset_results[dataset] = (
        weight_adjust,
        muE_test_unbiased,
        muE_test,
        muL_test_unbiased,
        muL_test_unadjusted,
        muL_test_adjusted,
    )

# Plotting
for dataset, (
    weight_adjust,
    muE_test_unbiased,
    muE_test,
    muL_test_unbiased,
    muL_test_unadjusted,
    muL_test_adjusted,
) in dataset_results.items():

    # Bar settings: 5 bars per cluster, centered on each metric's tick
    x = np.arange(len(measurable_metrics))  # x positions for metrics
    width = 0.15  # width of each bar (narrow enough to fit 5 bars)

    fig, ax = plt.subplots(figsize=(16, 6))

    colors_muE_test_unbiased = "red"
    colors_muE_test = "yellow"
    colors_muL_test_unbiased = "orange"
    colors_muL_test_unadjusted = "green"
    colors_muL_test_adjusted = "blue"

    # Clustered bars, evenly spaced around each x position
    ax.bar(x - 2*width, muE_test_unbiased, width,
           label=f'{selected_expert} Expert on Unbiased Data', color=colors_muE_test_unbiased)
    ax.bar(x - 1*width, muE_test, width,
           label=f'{selected_expert} Expert on Data With {biased_type[0]} Added', color=colors_muE_test)
    ax.bar(x, muL_test_unbiased, width,
           label='Fairness Reward Weight Optimization Model With No Weight Adjustment and No Bias', color=colors_muL_test_unbiased)
    ax.bar(x + 1*width, muL_test_unadjusted, width,
           label=f'Fairness Reward Weight Optimization Model With No Weight Adjustment on Data With {biased_type[0]} Added', color=colors_muL_test_unadjusted)
    ax.bar(x + 2*width, muL_test_adjusted, width,
           label=f'Fairness Reward Weight Optimization Model With {weight_adjust_label(weight_adjust)} Weight Adjustment on Data With {biased_type[0]} Added', color=colors_muL_test_adjusted)

    # Labels and title
    ax.set_ylabel('Learned Feature Expectations')
    ax.set_title(f"Learned Feature Expectations Throughout the Fairness Reward Weight Optimization Training Pipeline for the {dataset} Dataset")
    ax.set_xticks(x)
    ax.set_xticklabels(measurable_metrics, rotation=45, ha='right')
    ax.axhline(0, color='black', linewidth=0.8)
    ax.legend()

    # Add grid for better readability
    ax.grid(True, alpha=0.3, axis='y')

    plt.tight_layout()

    # Save and show
    plt.savefig(os.path.join(output_dir("exp_plots", WANDB_CONFIG, "training_pipeline"), f"Learned_Feat_Exp_of_Train_Pipeline_{dataset}_{slug(bias_type_label(biased_type))}.pdf"), bbox_inches="tight")
    plt.show()

In [ ]:
# Expert Preference Recovery (the expert_irl experiment config):

# Whether the learned Fairness Reward Weight Optimization classifier captures what each expert
# optimizes for. The expert_irl config learns from experts that optimize for different
# objectives (e.g. OptAcc for accuracy, HardtDemPar for demographic parity), and these figures
# compare what was learned from each:
# * `expert_irl_objectives_<dataset>.pdf`: one panel per expert, comparing the test feature
#   expectations of the expert's demonstrations (gray) with those of Fairness Reward Weight
#   Optimization learned from them, with its tuned weight adjustment (blue), and with the equal
#   reward weights its unadjusted run starts from (light blue), the same for every expert.
# * `expert_irl_weights_<dataset>.pdf`: the weights the weight adjustment learned from each
#   expert's demonstrations, one bar per expert under each objective, so that what each expert's
#   weights favor can be read across the experts. (The unadjusted run's weights are equal for
#   every expert, so they are not shown.)
# Each also comes as one figure with a row per dataset (`..._all_datasets.pdf`). Every bar is the
# mean over the trials, with a 95% confidence interval.
#
# The weights are L1-normalized and signed. Their objectives differ in scale and are correlated
# (DemPar, EqOpp and AccPar tend to move together), so compare one objective's weight across the
# experts rather than the weights of one expert across its objectives.
#
# This section loads its own runs, those of experiment config EXPERT_IRL_CONFIG, and does not
# use the main config's.

EXPERT_IRL_CONFIG = "expert_irl"
# None loads the most recent session that ran EXPERT_IRL_CONFIG.
EXPERT_IRL_SESSION = None
# The weight adjustment of each dataset's adjusted runs, if its runs have several, e.g.
#   {"COMPAS": ("opt_debias", "optuna", "TPE", 200)}
expert_irl_weight_adjust_override = {}
expert_irl_split = "test"

expert_irl = load_results(EXPERT_IRL_CONFIG, EXPERT_IRL_SESSION)
expert_irl_dir = output_dir("exp_plots", EXPERT_IRL_CONFIG)
expert_irl_experts = session_experts(expert_irl.averaged_data)
expert_irl_datasets = list(expert_irl.averaged_data)


def expert_irl_trials(dataset, expert):
    """The per-trial results of the unadjusted and the adjusted Fairness Reward Weight
    Optimization runs of one dataset and expert, and the adjusted runs' weight adjustment. None
    if the session did not run that expert on that dataset."""
    by_algorithm = expert_irl.per_trial_data[dataset].get(expert)
    if by_algorithm is None or () not in by_algorithm.get(weight_adjust_algorithm, {}):
        return None
    by_weight_adjust = by_algorithm[weight_adjust_algorithm][()]
    weight_adjust = single_weight_adjust(
        by_weight_adjust,
        f"dataset={dataset!r}, expert={expert!r} of W&B session {expert_irl.session_id!r}",
        expert_irl_weight_adjust_override.get(dataset),
    )
    return by_weight_adjust[()], by_weight_adjust[weight_adjust], weight_adjust


def plot_expert_irl_objectives(datasets, file_name):
    fig, axes = plt.subplots(
        len(datasets), len(expert_irl_experts),
        figsize=(2.4 * len(expert_irl_experts) + 0.4, 2.1 * len(datasets) + 0.5),
        squeeze=False, sharey=True,
    )
    for row, dataset in enumerate(datasets):
        for column, expert in enumerate(expert_irl_experts):
            ax = axes[row, column]
            trials = expert_irl_trials(dataset, expert)
            if trials is None:
                ax.set_visible(False)
                continue
            unadjusted, adjusted, weight_adjust = trials
            expert_fe = columns_of(unadjusted, f"muE_{expert_irl_split}_", "_mean")
            unadjusted_fe = columns_of(unadjusted, f"muL_{expert_irl_split}_")
            adjusted_fe = columns_of(adjusted, f"muL_{expert_irl_split}_")
            grouped_bars(ax, measurable_metrics, [
                ("Expert demonstrations", expert_fe.mean(), expert_fe.apply(ci95), EXPERT_COLOR),
                ("FRWO (equal weights)", unadjusted_fe.mean(), unadjusted_fe.apply(ci95),
                 FRWO_UNADJUSTED_COLOR),
                ("FRWO (learned weights)", adjusted_fe.mean(), adjusted_fe.apply(ci95), FRWO_COLOR),
            ])
            ax.set_ylim(0, 1.05)
            plt.setp(ax.get_xticklabels(), rotation=35, ha="right")
            if row == 0:
                ax.set_title(f"{expert} expert")
            if column == 0:
                ax.set_ylabel(f"{dataset_label(dataset)}\nFeature expectation")
    figure_title(fig, "Objectives of each expert's demonstrations and of FRWO learned from them")
    fig.tight_layout()
    shared_legend(fig, axes)
    save_figure(fig, os.path.join(expert_irl_dir, file_name))


def plot_expert_irl_weights(datasets, file_name):
    fig, axes = plt.subplots(len(datasets), 1, figsize=(4.6, 2.1 * len(datasets) + 0.6),
                             squeeze=False)
    for row, dataset in enumerate(datasets):
        ax = axes[row, 0]
        series = []
        for i, expert in enumerate(expert_irl_experts):
            trials = expert_irl_trials(dataset, expert)
            if trials is None:
                continue
            weights = columns_of(trials[1], "wL_")
            series.append((expert, weights.mean(), weights.apply(ci95),
                           SERIES_COLORS[i % len(SERIES_COLORS)]))
        if not series:
            ax.set_visible(False)
            continue
        grouped_bars(ax, measurable_metrics, series)
        ax.set_ylabel(f"{dataset_label(dataset)}\nLearned weight")
    figure_title(fig, "FRWO weights learned from each expert's demonstrations")
    fig.tight_layout()
    shared_legend(fig, axes, ncol=min(len(expert_irl_experts), 4))
    save_figure(fig, os.path.join(expert_irl_dir, file_name))


for dataset in expert_irl_datasets:
    plot_expert_irl_objectives([dataset], f"expert_irl_objectives_{dataset}.pdf")
    plot_expert_irl_weights([dataset], f"expert_irl_weights_{dataset}.pdf")
if len(expert_irl_datasets) > 1:
    plot_expert_irl_objectives(expert_irl_datasets, "expert_irl_objectives_all_datasets.pdf")
    plot_expert_irl_weights(expert_irl_datasets, "expert_irl_weights_all_datasets.pdf")

In [ ]:
# Controlling the Policy Through Its Weights (the manual_adjust experiment config):

# How Fairness Reward Weight Optimization's training adjustments (SUBDOM_WEIGHT_ADJUST_LIST and
# MANUAL_WEIGHT_ADJUST_LIST) change the learned weights and the objective values of the final
# policy. Each adjustment is compared with the plain run it adjusts, trial by trial:
# * A subdominance-weight adjustment skews only the `opt_debias` objective, so it is compared with
#   the plain `opt_debias` run.
# * A manual weight adjustment is added to the weights of the plain `opt_debias` run, or, with
#   `manual_adjust_base` "unadjusted", to the equal weights of the unadjusted run; it is compared
#   with that run.
#
# Figures, for every expert:
# * `<adjustment type>_<dataset>_<expert>.pdf`: one column per adjustment of the type's list; the
#   top row the weights and the bottom row the test objective values, before (gray) and after
#   (blue) the adjustment, as the mean over the trials with a 95% confidence interval. The
#   objective(s) an adjustment names are shaded and their tick labels bold. The objective values
#   carry the change in percentage points above each pair, and each column's title the change in
#   (unadjusted) test subdominance, so that the cost of steering away from the expert is shown too.
#   A list longer than `manual_adjust_max_columns` is split into one figure per objective the
#   adjustments name (`..._<objective>[_part<n>].pdf`); when that makes more than two figures per
#   dataset, they are only saved, not drawn here.
# * `<adjustment type>_dose_response_<expert>.pdf`, for a list that adjusts one objective by several
#   amounts (e.g. DemPar +0.1, +0.2, ... or DemPar ×2, ×4, ×8): one column per adjusted objective,
#   the change in every weight, every objective value and the subdominance against the amount
#   (multipliers on a log scale), as the mean over the datasets with a 95% confidence interval
#   across them. Each line starts from the unadjusted run (+0, or ×1), where every change is 0. The adjusted objective's line is the bold
#   one: that the adjustment controls the policy reads as it rising with the amount.
# * `<adjustment type>_summary_<expert>.pdf`, and `all_adjustments_summary_<expert>.pdf` for every
#   type at once: the mean change over the datasets (and trials) of each weight and of each
#   objective value, one row per adjustment, with the objective(s) it names outlined. That
#   adjustments control the policy reads as the outlined cells being the bluest of their rows.
#
# This section loads its own runs, those of experiment config MANUAL_ADJUST_CONFIG, and does not
# use the main config's.

MANUAL_ADJUST_CONFIG = "manual_adjust"
# None loads the most recent session that ran MANUAL_ADJUST_CONFIG.
MANUAL_ADJUST_SESSION = None
manual_adjust_base = "adjusted"  # or "unadjusted", for the manual weight adjustments only
# The `opt_debias` weight adjustment of each dataset's runs, if its runs have several, e.g.
#   {"COMPAS": ("opt_debias", "optuna", "TPE", 200)}
manual_adjust_weight_adjust_override = {}
manual_adjust_split = "test"
manual_adjust_bias_type = ()
manual_adjust_subdominance = "sum_abs_subdominance_test"
manual_adjust_max_columns = 5

ADJUST_KIND_TITLE = {
    "SUBDOM_WEIGHT_ADJUST": "Subdominance objective weight adjustment",
    "MANUAL_WEIGHT_ADJUST": "Manual weight adjustment",
}
ADJUST_KIND_SHORT = {
    "SUBDOM_WEIGHT_ADJUST": "Subdom. weight",
    "MANUAL_WEIGHT_ADJUST": "Manual",
}
ADJUST_KIND_FILE = {
    "SUBDOM_WEIGHT_ADJUST": "subdom_weight_adjust",
    "MANUAL_WEIGHT_ADJUST": "manual_weight_adjust",
}

manual_session_id, manual_records = read_session_runs(
    WANDB_PROJECT, WANDB_ENTITY, MANUAL_ADJUST_SESSION, MANUAL_ADJUST_CONFIG
)
manual_base = aggregate_records(manual_session_id, select_training_adjust(manual_records, None))
describe_results(manual_base)
# Every training adjustment the session ran, by type, in the order they were run
training_adjusts = {}
for record in sorted(manual_records, key=lambda record: record["created_at"]):
    training_adjust = run_training_adjust(record["config"])
    if training_adjust is not None:
        entries = training_adjusts.setdefault(training_adjust[0], [])
        if training_adjust[1] not in entries:
            entries.append(training_adjust[1])
# The runs of each training adjustment, in place of the runs they adjust
manual_adjusted = {
    (key, entry): aggregate_records(
        manual_session_id, select_training_adjust(manual_records, (key, entry))
    )
    for key, entries in training_adjusts.items()
    for entry in entries
}
for key, entries in training_adjusts.items():
    print(f"{key}: {list(entries)}")
manual_adjust_dir = output_dir("exp_plots", MANUAL_ADJUST_CONFIG)


def adjust_entry_label(key, entry):
    """E.g. `DemPar ×2` for a subdominance-weight adjustment, `DemPar +0.2` for a manual one."""
    if key == "MANUAL_WEIGHT_ADJUST":
        return ", ".join(f"{metric} {value:+g}" for metric, value in entry)
    return ", ".join(f"{metric} ×{value:g}" for metric, value in entry)


def adjustment_trials(key, entry, dataset, expert):
    """The per-trial results of the plain run and of the adjusted run, aligned on their trials, or
    None if the session did not run the adjustment for this dataset and expert."""
    by_weight_adjust = (
        manual_base.per_trial_data[dataset].get(expert, {}).get(weight_adjust_algorithm, {})
        .get(manual_adjust_bias_type)
    )
    if by_weight_adjust is None:
        return None
    base_weight_adjust = single_weight_adjust(
        by_weight_adjust,
        f"dataset={dataset!r}, expert={expert!r} of W&B session {manual_session_id!r}",
        manual_adjust_weight_adjust_override.get(dataset),
    )
    if key == "MANUAL_WEIGHT_ADJUST" and manual_adjust_base == "unadjusted":
        base_weight_adjust = ()
    adjusted = (
        manual_adjusted[(key, entry)].per_trial_data[dataset].get(expert, {})
        .get(weight_adjust_algorithm, {}).get(manual_adjust_bias_type, {}).get(base_weight_adjust)
    )
    if adjusted is None:
        return None
    before = by_weight_adjust[base_weight_adjust]
    trials = before.index.intersection(adjusted.index)
    return before.loc[trials], adjusted.loc[trials]


def base_label(key):
    if key == "MANUAL_WEIGHT_ADJUST" and manual_adjust_base == "unadjusted":
        return "Before (equal weights)"
    return "Before (plain opt_debias)"


def highlight_targets(ax, entry):
    """Shade the objectives `entry` names and make their tick labels bold."""
    targets = {metric for metric, _value in entry}
    for i, metric in enumerate(measurable_metrics):
        if metric in targets:
            ax.axvspan(i - 0.5, i + 0.5, color=GRIDLINE, alpha=0.6, zorder=0, linewidth=0)
    for tick in ax.get_xticklabels():
        if tick.get_text() in targets:
            tick.set_fontweight("bold")
            tick.set_color(INK)


def adjusted_metrics(entry):
    """The objectives an adjustment names, as a filename-friendly string."""
    return "_".join(metric for metric, _value in entry)


def plot_adjustment_type(key, dataset, expert):
    entries = [e for e in training_adjusts[key] if adjustment_trials(key, e, dataset, expert)]
    if not entries:
        return
    file_name = f"{ADJUST_KIND_FILE[key]}_{dataset}_{expert}"
    if len(entries) <= manual_adjust_max_columns:
        plot_adjustment_entries(key, dataset, expert, entries, f"{file_name}.pdf")
        return
    # One figure per adjusted objective, each in parts of at most manual_adjust_max_columns
    by_metrics = {}
    for entry in entries:
        by_metrics.setdefault(adjusted_metrics(entry), []).append(entry)
    figures = []
    for metrics, metric_entries in by_metrics.items():
        parts = [
            metric_entries[i:i + manual_adjust_max_columns]
            for i in range(0, len(metric_entries), manual_adjust_max_columns)
        ]
        for n, part in enumerate(parts, start=1):
            suffix = f"_part{n}" if len(parts) > 1 else ""
            figures.append((part, f"{file_name}_{metrics}{suffix}.pdf"))
    for part, part_file_name in figures:
        plot_adjustment_entries(
            key, dataset, expert, part, part_file_name, show=None if len(figures) <= 2 else False
        )


def plot_adjustment_entries(key, dataset, expert, entries, file_name, show=None):
    fig, axes = plt.subplots(2, len(entries), figsize=(2.5 * len(entries) + 0.5, 4.4),
                             squeeze=False, sharey="row")
    for column, entry in enumerate(entries):
        before, after = adjustment_trials(key, entry, dataset, expert)
        weights = (columns_of(before, "wL_"), columns_of(after, "wL_"))
        values = (
            columns_of(before, f"muL_{manual_adjust_split}_"),
            columns_of(after, f"muL_{manual_adjust_split}_"),
        )
        for row, (b, a) in enumerate((weights, values)):
            ax = axes[row, column]
            positions = grouped_bars(ax, measurable_metrics, [
                (base_label(key), b.mean(), b.apply(ci95), EXPERT_COLOR),
                ("After adjustment", a.mean(), a.apply(ci95), FRWO_COLOR),
            ])
            if row == 1:
                tops = np.maximum((b.mean() + b.apply(ci95)).to_numpy(),
                                  (a.mean() + a.apply(ci95)).to_numpy())
                annotate_differences(ax, np.mean(positions, axis=0), tops, (a - b).mean().to_numpy())
                ax.set_ylim(0, 1.1)
            plt.setp(ax.get_xticklabels(), rotation=35, ha="right")
            highlight_targets(ax, entry)
        subdominance_change = (
            after[manual_adjust_subdominance] - before[manual_adjust_subdominance]
        ).mean()
        axes[0, column].set_title(
            f"{adjust_entry_label(key, entry)}\nΔ subdominance {subdominance_change:+.3f}"
        )
    axes[0, 0].set_ylabel("Learned weight")
    axes[1, 0].set_ylabel(f"Feature expectation ({manual_adjust_split})")
    figure_title(fig, f"{ADJUST_KIND_TITLE[key]}: {dataset_label(dataset)}, {expert} expert")
    fig.tight_layout()
    shared_legend(fig, axes)
    save_figure(fig, os.path.join(manual_adjust_dir, file_name), show=show)


def adjustment_changes(adjustments, expert):
    """The mean change over the datasets and trials of every weight, objective value and of the
    subdominance, one row per `(key, entry)` of `adjustments`."""
    weight_changes, value_changes, subdominance_changes = [], [], []
    for key, entry in adjustments:
        by_dataset = [
            trials
            for dataset in manual_base.per_trial_data
            if (trials := adjustment_trials(key, entry, dataset, expert)) is not None
        ]
        weight_changes.append(np.nanmean(
            [(columns_of(a, "wL_") - columns_of(b, "wL_")).mean() for b, a in by_dataset], axis=0
        ) if by_dataset else np.full(len(measurable_metrics), np.nan))
        value_changes.append(np.nanmean([
            (columns_of(a, f"muL_{manual_adjust_split}_")
             - columns_of(b, f"muL_{manual_adjust_split}_")).mean()
            for b, a in by_dataset
        ], axis=0) * 100 if by_dataset else np.full(len(measurable_metrics), np.nan))
        subdominance_changes.append(np.nanmean([
            (a[manual_adjust_subdominance] - b[manual_adjust_subdominance]).mean()
            for b, a in by_dataset
        ]) if by_dataset else np.nan)
    return np.array(weight_changes), np.array(value_changes), np.array(subdominance_changes)


def plot_adjustment_summary(adjustments, expert, title, file_name):
    adjustments = [
        (key, entry) for key, entry in adjustments
        if any(adjustment_trials(key, entry, d, expert) for d in manual_base.per_trial_data)
    ]
    if not adjustments:
        return
    weight_changes, value_changes, subdominance_changes = adjustment_changes(adjustments, expert)
    several_kinds = len({key for key, _entry in adjustments}) > 1
    row_labels = [
        (f"{ADJUST_KIND_SHORT[key]}: " if several_kinds else "")
        + f"{adjust_entry_label(key, entry)}  (Δ subdom. {change:+.3f})"
        for (key, entry), change in zip(adjustments, subdominance_changes)
    ]
    fig, axes = plt.subplots(1, 2, figsize=(8.6, max(0.32 * len(adjustments) + 1.5, 2.6)),
                             gridspec_kw={"wspace": 0.35})
    for ax, changes, label, fmt in (
        (axes[0], weight_changes, "Change in learned weight", "{:+.2f}"),
        (axes[1], value_changes, "Change in feature expectation (pp)", "{:+.1f}"),
    ):
        image = diverging_heatmap(ax, changes, row_labels, measurable_metrics, annotate=True, fmt=fmt)
        # Outline the objectives each adjustment names
        for row, (_key, entry) in enumerate(adjustments):
            for metric, _value in entry:
                if metric in measurable_metrics:
                    column = measurable_metrics.index(metric)
                    ax.add_patch(plt.Rectangle((column - 0.5, row - 0.5), 1, 1, fill=False,
                                               edgecolor=INK, linewidth=1.2))
        colorbar_beside(fig, ax, image, label)
        ax.set_title(label)
    axes[1].set_yticklabels([])
    figure_title(fig, f"{title}: {expert} expert, mean over the datasets")
    # Room for the title above the panels' own
    fig.subplots_adjust(top=1 - 0.9 / fig.get_figheight())
    save_figure(fig, os.path.join(manual_adjust_dir, file_name))


def plot_dose_response(key, expert):
    """The change against the amount, for the objectives the list adjusts by several amounts."""
    by_metric = {}
    for entry in training_adjusts[key]:
        if len(entry) == 1:
            by_metric.setdefault(entry[0][0], []).append(entry)
    by_metric = {
        metric: sorted(entries, key=lambda entry: entry[0][1])
        for metric, entries in by_metric.items()
        if len(entries) > 1 and metric in measurable_metrics
    }
    if not by_metric:
        return
    fig, axes = plt.subplots(3, len(by_metric), figsize=(2.7 * len(by_metric) + 0.6, 6.4),
                             squeeze=False, sharex="col", sharey="row")
    multiplier = key != "MANUAL_WEIGHT_ADJUST"
    for column, (target, entries) in enumerate(by_metric.items()):
        # The unadjusted run first: adding 0, or multiplying by 1, changes nothing
        amounts = [1.0 if multiplier else 0.0] + [entry[0][1] for entry in entries]
        # Per entry, each dataset's mean change (rows: datasets)
        changes = {"weights": [], "values": [], "subdominance": []}
        for entry in entries:
            by_dataset = [
                trials for dataset in manual_base.per_trial_data
                if (trials := adjustment_trials(key, entry, dataset, expert)) is not None
            ]
            changes["weights"].append(np.array([
                (columns_of(a, "wL_") - columns_of(b, "wL_")).mean().to_numpy() for b, a in by_dataset
            ]))
            changes["values"].append(np.array([
                (columns_of(a, f"muL_{manual_adjust_split}_")
                 - columns_of(b, f"muL_{manual_adjust_split}_")).mean().to_numpy() * 100
                for b, a in by_dataset
            ]))
            changes["subdominance"].append(np.array([
                (a[manual_adjust_subdominance] - b[manual_adjust_subdominance]).mean()
                for b, a in by_dataset
            ]))
        for kind in changes:
            changes[kind].insert(0, np.zeros_like(changes[kind][0]))
        for row, kind in enumerate(("weights", "values")):
            ax = axes[row, column]
            for i, metric in enumerate(measurable_metrics):
                means = np.array([c[:, i].mean() if len(c) else np.nan for c in changes[kind]])
                errors = np.array([ci95(c[:, i]) if len(c) else 0 for c in changes[kind]])
                is_target = metric == target
                color = SERIES_COLORS[i % len(SERIES_COLORS)]
                ax.plot(amounts, means, color=color, linewidth=2.2 if is_target else 1.0,
                        marker="o" if is_target else None, markersize=4, label=metric,
                        zorder=4 if is_target else 3)
                if errors.max() > 0:
                    ax.fill_between(amounts, means - errors, means + errors, color=color,
                                    alpha=0.15 if is_target else 0.06, linewidth=0)
            ax.axhline(0, color=BASELINE, linewidth=0.8, zorder=2)
        ax = axes[2, column]
        means = np.array([c.mean() if len(c) else np.nan for c in changes["subdominance"]])
        errors = np.array([ci95(c) if len(c) else 0 for c in changes["subdominance"]])
        ax.plot(amounts, means, color=INK_SECONDARY, linewidth=1.6, marker="o", markersize=3,
                label="Subdominance")
        if errors.max() > 0:
            ax.fill_between(amounts, means - errors, means + errors, color=INK_SECONDARY,
                            alpha=0.12, linewidth=0)
        ax.axhline(0, color=BASELINE, linewidth=0.8, zorder=2)
        ax.set_xlabel(f"Multiplier of {target}'s subdominance weight" if multiplier
                      else f"Amount added to {target}")
        if multiplier:
            ax.set_xscale("log", base=2)
            ax.set_xticks(amounts)
            ax.set_xticklabels([f"×{amount:g}" for amount in amounts])
            ax.minorticks_off()
        axes[0, column].set_title(f"Adjusting {target}")
    axes[0, 0].set_ylabel("Change in learned weight")
    axes[1, 0].set_ylabel("Change in feature\nexpectation (pp)")
    axes[2, 0].set_ylabel("Change in subdominance")
    figure_title(fig, f"{ADJUST_KIND_TITLE[key]} by amount: {expert} expert, mean over the datasets")
    fig.tight_layout()
    shared_legend(fig, axes)
    save_figure(fig, os.path.join(manual_adjust_dir, f"{ADJUST_KIND_FILE[key]}_dose_response_{expert}.pdf"))


for expert in session_experts(manual_base.averaged_data):
    for key in training_adjusts:
        for dataset in manual_base.per_trial_data:
            plot_adjustment_type(key, dataset, expert)
        plot_dose_response(key, expert)
        plot_adjustment_summary(
            [(key, entry) for entry in training_adjusts[key]], expert, ADJUST_KIND_TITLE[key],
            f"{ADJUST_KIND_FILE[key]}_summary_{expert}.pdf",
        )
    if len(training_adjusts) > 1:
        plot_adjustment_summary(
            [(key, entry) for key, entries in training_adjusts.items() for entry in entries],
            expert, "Every training adjustment", f"all_adjustments_summary_{expert}.pdf",
        )